# IntelliRehabDS

Select a **GPU runtime** for the full profile. Run setup and the data checks, then the three numbered regimes. Progress shows the active model, Optuna trial, inner fold, and epoch. The individualized bar is quieter by default. Set `VERBOSE_INDIVIDUALIZED = True` for detailed messages there. After a disconnect, rerun the setup and helper cells, then resume with the same settings and run name.

For a short workflow check, set `PROFILE = "preview"`. It uses smaller training budgets and a separate output folder. Use `"full"` for manuscript results.

| Regime | Outer evaluation | Inner tuning |
|---|---|---|
| Generalized | Five grouped folds across all gestures | Subject-disjoint folds within outer training |
| Per gesture | Five grouped folds within each gesture, with eligible folds retained | Subject-disjoint folds within the gesture |
| Individualized | Thirty stratified holdouts per eligible participant–gesture pair | Up to three stratified folds within each holdout's training set |

The primary models are LR on flattened angle trajectories, DA-TCN on the sequences, and LR on 30 engineered angle descriptors. Each model receives the same held-out repetitions in a given regime.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys

PROJECT_ROOT = Path("/content/drive/MyDrive/NeuroYOLO_IRDS (1)")
DATA_ROOT = PROJECT_ROOT / "SkeletonData" / "SkeletonData" / "Simplified"
RESULTS_ROOT = PROJECT_ROOT / "clean_results"
RUN_NAME = "intellirehabds_three_regime_v2_gpu"
PROFILE = "full"
RUN_CROSS_METHOD = True  # RF, SVM, XGBoost, LSTM, with and without temporal shuffle
REQUIRE_GPU_FOR_FULL = True
VERBOSE_INDIVIDUALIZED = True

if PROFILE not in {"full", "preview"}:
    raise ValueError("PROFILE must be 'full' or 'preview'.")

try:
    from google.colab import drive
except ImportError:
    pass
else:
    drive.mount("/content/drive")

required_packages = ["optuna"] + (["xgboost"] if RUN_CROSS_METHOD else [])
missing = [name for name in required_packages if importlib.util.find_spec(name) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

PREVIEW = PROFILE == "preview"
SETTINGS = {
    "seed": 42,
    "outer_folds": 5,
    "group_inner_folds": 4,
    "individual_inner_folds": 3,
    "individual_repeats": 2 if PREVIEW else 30,
    "individual_test_fraction": 0.30,
    "min_test_per_class": 2,
    "lr_c_grid": [0.1, 1.0] if PREVIEW else [0.01, 0.03, 0.1, 0.3, 1.0, 3.0, 10.0],
    "da_trials_group": 1 if PREVIEW else 25,
    "da_trials_individual": 1 if PREVIEW else 10,
    "da_max_epochs": 1 if PREVIEW else 40,
    "da_min_epochs": 1 if PREVIEW else 10,
    "da_patience": 6,
    "phase_bins": 5,
    "robustness_seeds": 1 if PREVIEW else 10,
    "robustness_bootstrap": 1000,
    "comparison_bootstrap": 2000 if PREVIEW else 5000,
    "cross_method_repeats": 1 if PREVIEW else 10,
    "cross_method_lstm_epochs": 1 if PREVIEW else 40,
    "cross_method_trees": 10 if PREVIEW else 300,
    "cross_method": RUN_CROSS_METHOD,
    "individual_min_train_per_class": 2,
}
RUN_DIR = RESULTS_ROOT / (RUN_NAME + ("_preview" if PREVIEW else ""))


PROTOCOL_VERSION = "intellirehabds-v2.0"
CODE_SHA256 = "9802cc8fbfd8a51f55f4354367f32d6c79480b8f6d179077f85f3cf6b75becaf"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Setup and data checks

Each `Simplified/*.txt` file is one recording. The loader reads 75 comma-separated values per frame, reshapes them to **25 Kinect joints × 3 coordinates**, and uses the original Kinect-to-COCO17 map. It resamples the coordinates to 100 frames and computes the same ten angle trajectories in radians.

The filename supplies participant, session, gesture, repetition, correctness, and posture. Label **1 is correct**, label **2 is incorrect**, and the classifier uses 1 and 0 respectively. The original notebook reported **2,577 binary-labeled recordings** (2,047 correct and 530 incorrect), **30 participants**, and **nine gestures (0–8)**, after excluding 12 label-3 recordings. The data check shows the observed counts and saves the exclusions.

The internal `exercise` column contains the original gesture ID, so the output tables can be combined with REHAB24-6. Invalid recordings are reported before fitting. The raw folder is read directly each session, using the filename metadata.

In [ ]:
# Imports, reproducibility, metrics and file utilities

import hashlib
import importlib.metadata
import itertools
import json
import os
import platform
import random
import re
from contextlib import contextmanager
from dataclasses import dataclass
from uuid import uuid4
from zipfile import BadZipFile
from time import perf_counter

import joblib
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import scipy
import sklearn
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.stats import mannwhitneyu, wilcoxon
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, confusion_matrix, roc_auc_score, roc_curve
from sklearn.model_selection import GroupKFold, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm
try:
    from IPython.display import display
except ImportError:
    display = print

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cpu":
    torch.set_num_threads(min(4, os.cpu_count() or 1))
optuna.logging.set_verbosity(optuna.logging.WARNING)

T = 100
ANGLE_NAMES = [
    "L_elbow", "R_elbow", "L_knee", "R_knee", "L_shoulder", "R_shoulder",
    "L_hip", "R_hip", "torso_L_dev", "torso_R_dev",
]
FEATURE_NAMES = [
    f"{stat}_{angle}" for stat in ("rom", "mean_abs_d1", "mean_abs_d2")
    for angle in ANGLE_NAMES
]
METRICS = ("balanced_accuracy", "accuracy", "precision", "sensitivity", "specificity", "f1", "auc")
PRIMARY_MODELS = ("LR_angles", "DA_TCN", "LR_features")
REGIMES = ("generalized", "per_exercise", "individualized")
MODEL_LABELS = {"LR_angles": "LR (angles)", "DA_TCN": "DA-TCN", "LR_features": "LR (engineered)"}
COLORS = {"LR_angles": "#156f83", "DA_TCN": "#a4435e", "LR_features": "#796020"}
LABEL_COLORS = {0: "#b05a3c", 1: "#157785"}


def stable_seed(*parts):
    payload = json.dumps([SETTINGS["seed"], *parts], separators=(",", ":"))
    return int.from_bytes(hashlib.sha256(payload.encode()).digest()[:4], "big") & 0x7FFFFFFF


def seed_torch(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if DEVICE.type == "cuda":
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True


@contextmanager
def atomic_output(path):
    """Commit a result only after its temporary file was written successfully."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_name(f".{path.stem}-{uuid4().hex}{path.suffix}")
    try:
        yield temp
        temp.replace(path)
    finally:
        temp.unlink(missing_ok=True)


def save_csv(frame, path):
    with atomic_output(path) as temp:
        frame.to_csv(temp, index=False)


def save_json(obj, path):
    with atomic_output(path) as temp:
        temp.write_text(json.dumps(obj, indent=2, sort_keys=True, allow_nan=False), encoding="utf-8")


def save_npz(path, **arrays):
    with atomic_output(path) as temp:
        np.savez_compressed(temp, **arrays)


def save_figure(fig, stem):
    stem.parent.mkdir(parents=True, exist_ok=True)
    for extension in ("png", "pdf"):
        fig.savefig(stem.with_suffix("." + extension), dpi=240, bbox_inches="tight")
    plt.close(fig)


def binary_metrics(y, p, predicted=None):
    y = np.asarray(y, dtype=int)
    p = np.asarray(p, dtype=float)
    if not len(y) or y.shape != p.shape or not np.isin(y, (0, 1)).all():
        raise ValueError("Expected aligned nonempty binary labels and scores.")
    if not np.isfinite(p).all():
        raise ValueError("Nonfinite model scores.")
    h = (p >= 0.5).astype(int) if predicted is None else np.asarray(predicted, dtype=int)
    if h.shape != y.shape or not np.isin(h, (0, 1)).all():
        raise ValueError("Expected aligned binary predictions.")
    tn, fp, fn, tp = map(int, confusion_matrix(y, h, labels=[0, 1]).ravel())
    sensitivity = tp / (tp + fn) if tp + fn else np.nan
    specificity = tn / (tn + fp) if tn + fp else np.nan
    return {
        "n_test": len(y), "tn": tn, "fp": fp, "fn": fn, "tp": tp,
        "balanced_accuracy": (sensitivity + specificity) / 2,
        "accuracy": (tp + tn) / len(y),
        "precision": tp / (tp + fp) if tp + fp else 0.0,
        "sensitivity": sensitivity, "specificity": specificity,
        "f1": 2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else 0.0,
        "auc": roc_auc_score(y, p) if np.isin([0, 1], y).all() else np.nan,
    }


def signed_rank_p(differences):
    values = np.round(np.asarray(differences, dtype=float), 12)
    values = values[np.isfinite(values)]
    if len(values) < 2:
        return np.nan
    return float(wilcoxon(values, alternative="two-sided").pvalue) if np.any(values) else 1.0


def bh_adjust(values):
    values = np.asarray(values, dtype=float)
    result = np.full(values.size, np.nan)
    valid = np.flatnonzero(np.isfinite(values))
    if valid.size:
        order = valid[np.argsort(values[valid])]
        adjusted = values[order] * len(order) / np.arange(1, len(order) + 1)
        result[order] = np.minimum(1.0, np.minimum.accumulate(adjusted[::-1])[::-1])
    return result


def bootstrap_median(values, groups=None, *, seed=42):
    """Percentile CI for the median. Resample complete participant clusters if supplied."""
    values = np.asarray(values, dtype=float)
    finite = np.isfinite(values)
    values = values[finite]
    if groups is None:
        groups = np.arange(len(values))
    else:
        groups = np.asarray(groups)[finite]
    clusters = [values[groups == group] for group in np.unique(groups)]
    if len(clusters) < 2:
        return np.nan, np.nan
    rng = np.random.default_rng(seed)
    medians = [
        np.median(np.concatenate([clusters[i] for i in draw]))
        for draw in rng.integers(0, len(clusters), (SETTINGS["comparison_bootstrap"], len(clusters)))
    ]
    return tuple(np.quantile(medians, (0.025, 0.975)))


def exact_sign_flip_p(values):
    """Two-sided sign-flip sensitivity analysis on one difference per participant."""
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) < 2:
        return np.nan
    rng = np.random.default_rng(stable_seed("sign_flip"))
    signs = (np.array(list(itertools.product((-1, 1), repeat=len(values))))
             if len(values) <= 16 else rng.choice([-1, 1], (10000, len(values))))
    tail = np.count_nonzero(np.abs(signs @ values) >= abs(values.sum()) - 1e-12)
    return tail / len(signs) if len(values) <= 16 else (tail + 1) / (len(signs) + 1)


def group_summary(frame, keys, measures):
    long = frame.melt(id_vars=keys, value_vars=list(measures), var_name="metric", value_name="value")
    return (long.groupby([*keys, "metric"], dropna=False)["value"]
            .agg(n="count", mean="mean", sd="std", median="median", minimum="min", maximum="max")
            .reset_index())


def delta_summary(values, *, groups=None, seed=42):
    values = np.asarray(values, dtype=float)
    keep = np.isfinite(values)
    values = values[keep]
    groups = np.asarray(groups)[keep] if groups is not None else None
    lo, hi = bootstrap_median(values, groups, seed=seed)
    return {
        "n_units": len(values), "mean_delta": values.mean() if len(values) else np.nan,
        "median_delta": np.median(values) if len(values) else np.nan,
        "fraction_positive": np.mean(values > 0) if len(values) else np.nan,
        "wilcoxon_p_exploratory": signed_rank_p(values),
        "median_ci_low": lo, "median_ci_high": hi,
    }

hardware_name = torch.cuda.get_device_name(DEVICE) if DEVICE.type == "cuda" else "CPU"
print(f"Training device: {hardware_name}. Profile: {PROFILE}.")
if not PREVIEW and REQUIRE_GPU_FOR_FULL and DEVICE.type != "cuda":
    raise RuntimeError("Full profile expects a GPU runtime. Select GPU and rerun setup, "
                       "or set PROFILE='preview' for a short CPU check.")

_SHOW_EPOCHS = True

def _live(message, *, log=False, show=True):
    if show:
        print(message, flush=True)
    if log:
        with (RUN_DIR / "training_progress.log").open("a", encoding="utf-8") as file:
            print(message, file=file, flush=True)

Training device: NVIDIA L4. Profile: full.


In [ ]:
# IntelliRehabDS file loading, Kinect mapping, and angle extraction
KINECT_TO_COCO = {
    3: 0, 4: 5, 8: 6, 5: 7, 9: 8, 6: 9, 10: 10,
    12: 11, 16: 12, 13: 13, 17: 14, 14: 15, 18: 16,
}
REFERENCE_COUNTS = {
    "files": 2589, "repetitions": 2577, "participants": 30, "gestures": 9,
    "correct": 2047, "incorrect": 530, "excluded_labels": 12,
}
META_COLUMNS = [
    "trial_id", "filename", "subject", "session", "exercise",
    "repetition", "correctness", "posture", "n_frames", "y",
]


def parse_intelli_filename(path):
    """Parse subject_session_gesture_repetition_correctness_posture.txt."""
    parts = path.stem.split("_", 5)
    if len(parts) != 6 or not parts[-1]:
        raise ValueError("expected subject_session_gesture_repetition_correctness_posture.txt")
    subject, session, gesture, repetition, correctness = map(int, parts[:5])
    return {
        "trial_id": path.name, "filename": path.name, "subject": subject,
        "session": session, "exercise": gesture, "repetition": repetition,
        "correctness": correctness, "posture": parts[-1],
    }


def load_intelli_skeleton(path):
    """Each text row is a frame of 25 Kinect joints with three coordinates each."""
    raw = np.loadtxt(path, delimiter=",", ndmin=2)
    if raw.shape[0] < 2 or raw.shape[1] != 75:
        raise ValueError(f"expected at least two frames with 75 values each, got {raw.shape}")
    if not np.isfinite(raw).all():
        raise ValueError("coordinates contain NaN or infinity")
    return raw.reshape(len(raw), 25, 3)


def load_intellirehabds(root):
    """Read the existing Simplified directory in the original sorted filename order."""
    root = Path(root)
    if not root.is_dir():
        raise FileNotFoundError(f"Set DATA_ROOT to your existing IntelliRehabDS Simplified folder. Missing: {root}")
    paths = sorted(root.glob("*.txt"))
    if not paths:
        raise FileNotFoundError(f"No .txt skeleton recordings directly inside {root}")

    rows, angles, coordinates, exclusions, errors = [], [], [], [], []
    source_joints, target_joints = list(KINECT_TO_COCO), list(KINECT_TO_COCO.values())
    for path in tqdm(paths, desc="Loading IntelliRehabDS", unit="recording"):
        try:
            row = parse_intelli_filename(path)
            if row["correctness"] not in (1, 2):
                exclusions.append({**row, "reason": "label outside the original binary cohort"})
                continue
            skeleton = load_intelli_skeleton(path)
            mapped = np.zeros((len(skeleton), 17, 3), dtype=np.float32)
            mapped[:, target_joints] = skeleton[:, source_joints]
            coco = resample_frames(mapped)
            values = compute_angles(coco)
            if values.shape != (T, len(ANGLE_NAMES)) or not np.isfinite(values).all():
                raise ValueError(f"invalid angle tensor {values.shape}")
            rows.append({**row, "n_frames": len(skeleton), "y": int(row["correctness"] == 1)})
            angles.append(values)
            coordinates.append(coco)
        except (OSError, ValueError, OverflowError) as error:
            errors.append({"filename": path.name, "error": f"{type(error).__name__}: {error}"})

    if errors:
        error_path = RUN_DIR.parent / "loading_audit" / RUN_DIR.name / "loading_errors.csv"
        report = pd.DataFrame(errors)
        save_csv(report, error_path)
        print(report.head(10).to_string(index=False, max_colwidth=None))
        raise RuntimeError(f"{len(errors)} recording errors. See {error_path}")
    if not rows:
        raise ValueError("No labeled recordings with correctness 1 or 2 were found.")

    meta = pd.DataFrame(rows, columns=META_COLUMNS)
    if meta.trial_id.duplicated().any():
        raise ValueError("Duplicate IntelliRehabDS filenames.")
    excluded = pd.DataFrame(exclusions, columns=[
        "trial_id", "filename", "subject", "session", "exercise", "repetition",
        "correctness", "posture", "reason",
    ])
    counts = {
        "files": len(paths), "repetitions": len(meta), "participants": int(meta.subject.nunique()),
        "gestures": int(meta.exercise.nunique()), "correct": int(meta.y.sum()),
        "incorrect": int((meta.y == 0).sum()), "excluded_labels": len(excluded),
    }
    differences = [f"{k}={v} (reference {REFERENCE_COUNTS[k]})"
                   for k, v in counts.items() if v != REFERENCE_COUNTS[k]]
    if differences:
        print("Cohort differs from the original notebook: " + ", ".join(differences))
    return meta, np.stack(angles), np.stack(coordinates), excluded, counts


def initialize_run(meta, angles, coco, excluded, counts):
    digest = hashlib.sha256(meta.to_csv(index=False).encode())
    digest.update(excluded.to_csv(index=False).encode())
    digest.update(angles.tobytes())
    digest.update(coco.tobytes())
    packages = ["numpy", "pandas", "scipy", "scikit-learn", "torch", "optuna", "joblib"]
    if RUN_CROSS_METHOD:
        packages.append("xgboost")
    manifest = {
        "dataset": "IntelliRehabDS", "profile": PROFILE, "protocol_version": PROTOCOL_VERSION,
        "code_sha256": CODE_SHA256, "data_sha256": digest.hexdigest(),
        "config": SETTINGS, "device_type": DEVICE.type, "python": platform.python_version(),
        "versions": {p: importlib.metadata.version(p) for p in packages},
        "source_label_definition": {"1": "correct", "2": "incorrect", "3": "excluded"},
        "label_definition": {"0": "incorrect", "1": "correct"},
        "angle_names": ANGLE_NAMES, "n_trials": len(meta),
        "n_participants": int(meta.subject.nunique()), "n_exercises": int(meta.exercise.nunique()),
    }
    path = RUN_DIR / "manifest.json"
    if path.exists():
        previous = json.loads(path.read_text(encoding="utf-8"))
        changed = [k for k, v in manifest.items() if previous.get(k) != v]
        if changed:
            raise RuntimeError(f"Existing run differs in {changed}. Set a new RUN_NAME to start a separate run.")
        print(f"Resuming {RUN_DIR}")
    else:
        save_json(manifest, path)
    save_csv(meta, RUN_DIR / "data_audit" / "metadata.csv")
    save_csv(excluded, RUN_DIR / "data_audit" / "excluded_labels.csv")
    cohort = pd.DataFrame([
        {"field": k, "observed": v, "reference": REFERENCE_COUNTS[k]} for k, v in counts.items()
    ])
    save_csv(cohort, RUN_DIR / "data_audit" / "cohort_counts.csv")
    print(f"{len(meta):,} repetitions, {meta.subject.nunique()} participants, "
          f"{meta.exercise.nunique()} gestures. Angle tensor {angles.shape}.")
    print(f"Correct: {counts['correct']:,}. Incorrect: {counts['incorrect']:,}. "
          f"Excluded labels: {len(excluded)}.")
    print(f"Results: {RUN_DIR}")


def resample_frames(seq, target=T):
    if len(seq) == target:
        return seq.astype(np.float32, copy=False)
    positions = np.linspace(0, len(seq) - 1, target)
    lo = np.floor(positions).astype(int)
    hi = np.minimum(lo + 1, len(seq) - 1)
    weights = (positions - lo).astype(np.float32)[:, None, None]
    return ((1 - weights) * seq[lo] + weights * seq[hi]).astype(np.float32)


def compute_angles(coco):
    """Original IntelliRehabDS ten angles, in radians and in ANGLE_NAMES order."""
    if coco.ndim != 3 or coco.shape[1:] != (17, 3):
        raise ValueError(f"Expected (frames, 17, 3) COCO coordinates, got {coco.shape}")

    def angle(a, b, c):
        u, v = coco[:, a] - coco[:, b], coco[:, c] - coco[:, b]
        denom = np.linalg.norm(u, axis=-1) * np.linalg.norm(v, axis=-1)
        return np.arccos(np.clip(np.sum(u * v, axis=-1) / (denom + 1e-8), -1.0, 1.0))

    def unit(v):
        return v / (np.linalg.norm(v, axis=-1, keepdims=True) + 1e-8)

    ls, rs, lh, rh = [coco[:, j] for j in (5, 6, 11, 12)]
    mid = unit((ls + rs) / 2.0 - (lh + rh) / 2.0)
    torso = [np.arccos(np.clip(np.sum(unit(v) * mid, axis=-1), -1.0, 1.0))
             for v in (ls - lh, rs - rh)]
    return np.stack([
        angle(5, 7, 9), angle(6, 8, 10), angle(11, 13, 15), angle(12, 14, 16),
        angle(7, 5, 11), angle(8, 6, 12), angle(5, 11, 13), angle(6, 12, 14), *torso,
    ], axis=1).astype(np.float32)


def engineered_features(x):
    """Original 30 engineered features. D1 and D2 use normalized time steps."""
    return np.concatenate([np.ptp(x, axis=1), np.abs(np.diff(x, axis=1)).mean(axis=1),
                           np.abs(np.diff(x, n=2, axis=1)).mean(axis=1)], axis=1)

In [ ]:
# Shared outer and inner split plans with leakage checks

@dataclass(frozen=True)
class Split:
    regime: str
    unit: str
    train: np.ndarray
    test: np.ndarray
    inner: tuple
    fold: int | None = None
    subject: int | None = None
    exercise: int | None = None
    repeat: int | None = None

    def keys(self):
        return {"regime": self.regime, "unit": self.unit, "fold": self.fold,
                "subject": self.subject, "exercise": self.exercise, "repeat": self.repeat}


def make_split_plan(meta):
    y = meta.y.to_numpy()
    subjects = meta.subject.to_numpy()
    plan = {regime: [] for regime in REGIMES}
    exclusions = []
    for regime in ("generalized", "per_exercise"):
        groups = [(None, meta)] if regime == "generalized" else meta.groupby("exercise", sort=True)
        for exercise, group in groups:
            ex = None if exercise is None else int(exercise)
            indices = group.index.to_numpy()
            n_groups = len(np.unique(subjects[indices]))
            if n_groups < SETTINGS["outer_folds"]:
                exclusions.append({"regime": regime, "exercise": ex, "reason": "too few outer subject groups"})
                continue
            outer = GroupKFold(n_splits=SETTINGS["outer_folds"])
            for fold, (a, b) in enumerate(outer.split(indices, y[indices], subjects[indices]), 1):
                train, test = indices[a], indices[b]
                unit = f"fold{fold:02d}" if ex is None else f"ex{ex:02d}_fold{fold:02d}"
                details = {"regime": regime, "unit": unit, "fold": fold, "exercise": ex}
                if min(np.bincount(y[train], minlength=2)) == 0 or min(np.bincount(y[test], minlength=2)) == 0:
                    exclusions.append({**details, "reason": "one class in outer train or test"})
                    continue
                k = min(SETTINGS["group_inner_folds"], len(np.unique(subjects[train])))
                inner = []
                if k >= 2:
                    cv = GroupKFold(n_splits=k)
                    for inner_fold, (a, b) in enumerate(cv.split(train, y[train], subjects[train]), 1):
                        tr, val = train[a], train[b]
                        if len(np.unique(y[tr])) == len(np.unique(y[val])) == 2:
                            inner.append((tr, val))
                        else:
                            exclusions.append({**details, "inner_fold": inner_fold,
                                               "reason": "one class in inner train or validation"})
                if len(inner) < 2:
                    exclusions.append({**details, "reason": "fewer than two valid grouped inner folds"})
                    continue
                plan[regime].append(Split(regime, unit, train, test, tuple(inner), fold=fold, exercise=ex))

    test_min = SETTINGS["min_test_per_class"]
    train_min = SETTINGS["individual_min_train_per_class"]
    max_inner = SETTINGS["individual_inner_folds"]
    for (subject, exercise), group in meta.groupby(["subject", "exercise"], sort=True):
        subject, exercise = int(subject), int(exercise)
        indices = group.index.to_numpy()
        classes = [indices[y[indices] == label] for label in (0, 1)]
        counts = [len(g) for g in classes]
        if min(counts) < test_min + train_min:
            exclusions.append({"regime": "individualized", "subject": subject, "exercise": exercise,
                               "n_incorrect": counts[0], "n_correct": counts[1],
                               "reason": f"need {test_min} test and {train_min} train per class"})
            continue
        test_counts = [min(n - train_min, max(test_min, round(n * SETTINGS["individual_test_fraction"])))
                       for n in counts]
        for repeat in range(1, SETTINGS["individual_repeats"] + 1):
            unit = f"s{subject:03d}_ex{exercise:02d}_r{repeat:02d}"
            rng = np.random.default_rng(stable_seed("outer", unit))
            test = np.sort(np.concatenate([rng.choice(g, size=k, replace=False)
                                            for g, k in zip(classes, test_counts)]))
            train = np.setdiff1d(indices, test)
            # Keep the sparse pairs that still support two inner folds.
            inner_k = min(max_inner, int(np.bincount(y[train], minlength=2).min()))
            cv = StratifiedKFold(n_splits=inner_k, shuffle=True, random_state=stable_seed("inner", unit))
            inner = tuple((train[a], train[b]) for a, b in cv.split(train, y[train]))
            plan["individualized"].append(Split("individualized", unit, train, test, inner,
                                                  subject=subject, exercise=exercise, repeat=repeat))
    return plan, pd.DataFrame(exclusions, columns=[
        "regime", "unit", "fold", "inner_fold", "subject", "exercise",
        "n_incorrect", "n_correct", "reason",
    ])


def audit_splits(meta, plan, exclusions):
    rows, outer_rows, inner_rows = [], [], []
    subjects, labels, ids = (meta[key].to_numpy() for key in ("subject", "y", "trial_id"))
    for regime, splits in plan.items():
        for split in splits:
            train, test = split.train, split.test
            if np.intersect1d(train, test).size:
                raise ValueError(f"Train/test trial overlap in {split.unit}")
            if len(np.unique(labels[train])) != 2 or len(np.unique(labels[test])) != 2:
                raise ValueError(f"Missing outer class in {split.unit}")
            group_disjoint = regime != "individualized"
            if group_disjoint and np.intersect1d(subjects[train], subjects[test]).size:
                raise ValueError(f"Train/test subject overlap in {split.unit}")
            rows.append({**split.keys(), "n_train": len(train), "n_test": len(test),
                         "train_n0": int((labels[train] == 0).sum()), "train_n1": int((labels[train] == 1).sum()),
                         "test_n0": int((labels[test] == 0).sum()), "test_n1": int((labels[test] == 1).sum()),
                         "n_train_subjects": len(np.unique(subjects[train])),
                         "n_test_subjects": len(np.unique(subjects[test])), "n_inner": len(split.inner)})
            for subset, ix in (("train", train), ("test", test)):
                outer_rows.extend({"regime": regime, "unit": split.unit, "subset": subset, "trial_id": ids[i]}
                                  for i in ix)
            for j, (tr, val) in enumerate(split.inner, 1):
                if np.intersect1d(tr, val).size or not np.array_equal(np.union1d(tr, val), np.sort(train)):
                    raise ValueError(f"Invalid inner split {split.unit}, fold {j}")
                if group_disjoint and np.intersect1d(subjects[tr], subjects[val]).size:
                    raise ValueError(f"Inner subject overlap in {split.unit}, fold {j}")
                if len(np.unique(labels[tr])) != 2 or len(np.unique(labels[val])) != 2:
                    raise ValueError(f"Missing inner class in {split.unit}, fold {j}")
                inner_rows.extend({"regime": regime, "unit": split.unit, "inner_fold": j, "trial_id": ids[i]}
                                  for i in val)
    audit = pd.DataFrame(rows)
    if audit.empty:
        raise ValueError("No eligible splits. Inspect the loaded data and label counts.")
    save_csv(audit, RUN_DIR / "splits" / "unit_counts.csv")
    save_csv(exclusions, RUN_DIR / "splits" / "exclusions.csv")
    save_csv(pd.DataFrame(outer_rows), RUN_DIR / "splits" / "outer_membership.csv")
    save_csv(pd.DataFrame(inner_rows), RUN_DIR / "splits" / "inner_validation_membership.csv")
    pair_count = audit.loc[audit.regime == "individualized", ["subject", "exercise"]].drop_duplicates().shape[0]
    counts = ", ".join(f"{name} {len(splits):,}" for name, splits in plan.items())
    print(f"Eligible splits: {counts}. Individualized participant–gesture pairs: {pair_count}.")
    print(f"Logged exclusions: {len(exclusions)}. Preview: {PREVIEW}.")
    return audit

In [ ]:
meta, X, coco, excluded_labels, cohort_counts = load_intellirehabds(DATA_ROOT)
initialize_run(meta, X, coco, excluded_labels, cohort_counts)

counts = meta.groupby(["subject", "exercise", "y"]).size().rename("n").reset_index()
save_csv(counts, RUN_DIR / "data_audit" / "class_counts.csv")
tensor_path = RUN_DIR / "data_audit" / "intellirehabds_primary_angles.npz"
if not tensor_path.exists():
    save_npz(tensor_path, X=X, y=meta.y.to_numpy(dtype=np.int8),
             subject_id=meta.subject.to_numpy(dtype=str), task_id=meta.exercise.to_numpy(dtype=str),
             sample_id=meta.trial_id.to_numpy(dtype=str), angle_names=np.asarray(ANGLE_NAMES))

plan, exclusions = make_split_plan(meta)
split_audit = audit_splits(meta, plan, exclusions)
display(split_audit.groupby("regime").agg(n_splits=("unit", "count"),
    min_train=("n_train", "min"), max_train=("n_train", "max"),
    min_test=("n_test", "min"), max_test=("n_test", "max")))
display(split_audit[split_audit.regime == "per_exercise"]
        .groupby("exercise").size().rename("eligible_outer_folds").to_frame())
lr_inputs = {"LR_angles": X.reshape(len(X), -1), "LR_features": engineered_features(X)}
results = {}

Loading IntelliRehabDS:   0%|          | 0/2589 [00:00<?, ?recording/s]

Resuming /content/drive/MyDrive/NeuroYOLO_IRDS (1)/clean_results/intellirehabds_three_regime_v2_gpu
2,577 repetitions, 30 participants, 9 gestures. Angle tensor (2577, 100, 10).
Correct: 2,047. Incorrect: 530. Excluded labels: 12.
Results: /content/drive/MyDrive/NeuroYOLO_IRDS (1)/clean_results/intellirehabds_three_regime_v2_gpu
Eligible splits: generalized 5, per_exercise 42, individualized 480. Individualized participant–gesture pairs: 16.
Logged exclusions: 248. Preview: False.


,n_splits,min_train,max_train,min_test,max_test
regime,,,,,
generalized,5,2057,2069,508,520
individualized,480,6,45,4,19
per_exercise,42,204,302,49,77


,eligible_outer_folds
exercise,
0.0,5
1.0,5
2.0,5
3.0,5
4.0,5
5.0,4
6.0,5
7.0,4
8.0,4


### Model fitting and result helpers

LR and DA-TCN use the same inner partitions and select configurations by mean validation balanced accuracy. LR fits its scaler on each training partition. DA-TCN selects the epoch on inner validation and refits on the complete outer training set for the rounded-up median of the selected epochs. The full profile uses 25 Optuna candidates per cross-subject fold and 10 per individualized repeat.

Progress reports model and tuning status. A finished fit has its own checkpoint. DA-TCN also saves tuning after each completed inner fold in the cross-subject regimes and after each completed trial in the individualized regime. To resume after a disconnect, keep the same run name, profile, device type, and settings, and run the desired regime again.

The metric tables include precision, sensitivity, specificity, F1, balanced accuracy, accuracy, ROC-AUC, and confusion counts. The positive class is correct execution and the primary decision threshold is 0.5. Precision and F1 are zero when their denominator is zero.

In [ ]:
# LR, DA-TCN, shared inner selection and linear SHAP

class TemporalBlock(nn.Module):
    """Two causal dilated convolutions with a same-length residual."""
    def __init__(self, channels, kernel, dilation, dropout):
        super().__init__()
        self.left_pad = (kernel - 1) * dilation
        self.conv1 = nn.Conv1d(channels, channels, kernel, dilation=dilation)
        self.conv2 = nn.Conv1d(channels, channels, kernel, dilation=dilation)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        y = self.dropout(F.relu(self.conv1(F.pad(x, (self.left_pad, 0)))))
        y = self.dropout(F.relu(self.conv2(F.pad(y, (self.left_pad, 0)))))
        return F.relu(x + y)


class DualAttentionTCN(nn.Module):
    def __init__(self, params):
        super().__init__()
        channels = params["hidden"]
        n_angles = len(ANGLE_NAMES)
        self.beta_mlp = nn.Sequential(nn.Linear(n_angles, 32), nn.ReLU(), nn.Linear(32, n_angles))
        self.projection = nn.Conv1d(n_angles, channels, 1)
        self.tcn = nn.Sequential(*[
            TemporalBlock(channels, params["kernel"], 2 ** i, params["dropout"])
            for i in range(params["levels"])
        ])
        self.alpha_head = nn.Linear(channels, 1)
        self.classifier = nn.Sequential(nn.Linear(channels, 64), nn.ReLU(),
                                        nn.Dropout(params["dropout"]), nn.Linear(64, 2))

    def forward(self, angles):
        beta = torch.softmax(self.beta_mlp(angles.mean(dim=1)), dim=1)
        gated = (angles * beta.unsqueeze(1)).transpose(1, 2)
        encoded = self.tcn(self.projection(gated)).transpose(1, 2)
        alpha = torch.softmax(self.alpha_head(encoded).squeeze(-1), dim=1)
        pooled = (encoded * alpha.unsqueeze(-1)).sum(dim=1)
        return self.classifier(pooled), alpha, beta


@torch.inference_mode()
def predict_da(model, x):
    model.eval()
    data = torch.as_tensor(x, dtype=torch.float32, device=DEVICE)
    probabilities, alphas, betas = [], [], []
    for batch in data.split(128):
        logits, alpha, beta = model(batch)
        probabilities.append(torch.softmax(logits, dim=1)[:, 1].cpu().numpy())
        alphas.append(alpha.cpu().numpy())
        betas.append(beta.cpu().numpy())
    return tuple(np.concatenate(arrays) for arrays in (probabilities, alphas, betas))


def train_da(x, y, params, seed, epochs, validation=None):
    seed_torch(seed)
    model = DualAttentionTCN(params).to(DEVICE)
    x_t = torch.as_tensor(x, dtype=torch.float32, device=DEVICE)
    y_t = torch.as_tensor(y, dtype=torch.long, device=DEVICE)
    counts = np.bincount(y, minlength=2)
    weights = torch.as_tensor(len(y) / (2 * counts), dtype=torch.float32, device=DEVICE)
    loss_fn = nn.CrossEntropyLoss(weight=weights)
    optimizer = torch.optim.AdamW(model.parameters(), lr=params["lr"], weight_decay=params["wd"])
    generator = torch.Generator(device=DEVICE).manual_seed(seed)
    best_score, best_loss, best_epoch, patience_count = -np.inf, np.inf, epochs, 0

    if validation is not None:
        x_val, y_val = validation
        val_tensor = torch.as_tensor(x_val, dtype=torch.float32, device=DEVICE)
        val_target = torch.as_tensor(y_val, dtype=torch.long, device=DEVICE)
        val_weights = torch.as_tensor(
            len(y_val) / (2 * np.bincount(y_val, minlength=2)),
            dtype=torch.float32, device=DEVICE,
        )

    started = last_message = perf_counter()
    phase = "inner fit" if validation is not None else "outer refit"

    for epoch in range(1, epochs + 1):
        now = perf_counter()
        if (_SHOW_EPOCHS and (epoch == 1 or epoch % 5 == 0 or epoch == epochs)
                or now - last_message >= 45):
            best = f" | best validation BA {best_score:.3f}" if np.isfinite(best_score) else ""
            _live(f"    {phase}: starting epoch {epoch}/{epochs} | {now - started:.0f}s elapsed{best}")
            last_message = now

        model.train()
        order = torch.randperm(len(y), generator=generator, device=DEVICE)
        for batch in order.split(params["batch"]):
            optimizer.zero_grad(set_to_none=True)
            logits, _, _ = model(x_t[batch])
            loss = loss_fn(logits, y_t[batch])
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        if validation is not None:
            model.eval()
            with torch.inference_mode():
                logits = torch.cat([model(chunk)[0] for chunk in val_tensor.split(128)])
                predictions = (logits[:, 1] >= logits[:, 0]).cpu().numpy()
                val_loss = F.cross_entropy(logits, val_target, weight=val_weights).item()
            score = balanced_accuracy_score(y_val, predictions)
            tied_and_lower_loss = abs(score - best_score) < 1e-10 and val_loss < best_loss - 1e-5
            if score > best_score + 1e-10 or tied_and_lower_loss:
                best_score, best_loss, best_epoch, patience_count = score, val_loss, epoch, 0
            else:
                patience_count += 1
            if epoch >= SETTINGS["da_min_epochs"] and patience_count >= SETTINGS["da_patience"]:
                break

    return model, best_epoch, best_score


def make_lr(c):
    return make_pipeline(StandardScaler(), LogisticRegression(
        C=float(c), class_weight="balanced", solver="liblinear", max_iter=4000,
        random_state=SETTINGS["seed"],
    ))


def tune_lr(features, y, split):
    records = []
    grid = SETTINGS["lr_c_grid"]
    for number, c in enumerate(grid, 1):
        started = perf_counter()
        _live(f"  LR grid {number}/{len(grid)} | C={c} | {len(split.inner)} inner folds",
              show=_SHOW_EPOCHS)
        scores = []
        for train, val in split.inner:
            model = make_lr(c).fit(features[train], y[train])
            predicted = (model.predict_proba(features[val])[:, 1] >= 0.5).astype(int)
            scores.append(float(balanced_accuracy_score(y[val], predicted)))
        records.append({
            "candidate": number, "params": {"C": float(c)},
            "inner_balanced_accuracy": float(np.mean(scores)), "inner_fold_scores": scores,
        })
        _live(f"    mean BA {np.mean(scores):.3f} | {perf_counter() - started:.0f}s",
              show=_SHOW_EPOCHS)

    best = max(records, key=lambda row: (row["inner_balanced_accuracy"], -row["params"]["C"]))
    return best, records


def da_search_space(trial, individualized):
    if PREVIEW:
        return {"hidden": 8, "levels": 2, "kernel": 3, "dropout": 0.1,
                "batch": 8, "lr": 0.001, "wd": 0.0001}
    return {
        "hidden": trial.suggest_categorical("hidden", [32, 64, 96, 128]),
        "levels": trial.suggest_categorical("levels", [3, 4, 5]),
        "kernel": trial.suggest_categorical("kernel", [3, 5, 7]),
        "dropout": trial.suggest_float("dropout", 0.05, 0.35),
        "batch": trial.suggest_categorical("batch", [4, 8, 16] if individualized else [32, 64, 128]),
        "lr": trial.suggest_float("lr", 1e-4, 3e-3, log=True),
        "wd": trial.suggest_float("wd", 1e-6, 1e-3, log=True),
    }


def tune_da(angles, y, split):
    individualized = split.regime == "individualized"
    budget = SETTINGS["da_trials_individual" if individualized else "da_trials_group"]
    candidates = []
    path = checkpoint_path(split, "DA_TCN").with_suffix(".tuning.json")
    signature = hashlib.sha256(json.dumps({
        "code": CODE_SHA256, "settings": SETTINGS,
        "inner": [[tr.tolist(), va.tolist()] for tr, va in split.inner],
    }, sort_keys=True).encode()).hexdigest()

    journal = {"signature": signature, "trials": {}}
    if path.exists():
        try:
            previous = json.loads(path.read_text(encoding="utf-8"))
            if (isinstance(previous, dict) and previous.get("signature") == signature
                    and isinstance(previous.get("trials"), dict)):
                journal = previous
        except (OSError, ValueError, KeyError) as error:
            _live(f"  Could not read tuning checkpoint for {split.unit}: {error}")

    def objective(trial):
        number = trial.number + 1
        params = da_search_space(trial, individualized)
        key = str(number)
        cached = journal["trials"].get(key, {})
        if (cached.get("params") == params and "scores" in cached and "epochs" in cached
                and len(cached["scores"]) == len(cached["epochs"]) <= len(split.inner)):
            scores, epochs = cached["scores"], cached["epochs"]
        else:
            scores, epochs = [], []
            journal["trials"][key] = {"params": params, "scores": scores, "epochs": epochs}

        label = f"  DA-TCN {split.unit} | trial {number}/{budget}"
        _live(f"{label} | {len(scores)}/{len(split.inner)} inner folds saved", show=_SHOW_EPOCHS)

        for j in range(len(scores) + 1, len(split.inner) + 1):
            train, val = split.inner[j - 1]
            started = perf_counter()
            _live(f"    inner fold {j}/{len(split.inner)} | train={len(train)} | validation={len(val)}",
                  show=_SHOW_EPOCHS)
            _, best_epoch, score = train_da(
                angles[train], y[train], params, stable_seed("da_inner", split.unit, j),
                SETTINGS["da_max_epochs"], validation=(angles[val], y[val]),
            )
            scores.append(float(score))
            epochs.append(int(best_epoch))
            if not individualized:
                save_json(journal, path)
            _live(f"    inner fold {j} done | BA {score:.3f} | best epoch {best_epoch}"
                  f" | {perf_counter() - started:.0f}s", show=_SHOW_EPOCHS)

        # Individualized trials are smaller, so commit once per trial.
        if individualized:
            save_json(journal, path)

        row = {
            "candidate": number, "params": params,
            "inner_balanced_accuracy": float(np.mean(scores)),
            "inner_fold_scores": list(scores), "inner_best_epochs": list(epochs),
            "refit_epochs": max(1, int(np.ceil(np.median(epochs)))),
        }
        candidates.append(row)
        best_so_far = max(r["inner_balanced_accuracy"] for r in candidates)
        _live(f"{label} done | mean BA {row['inner_balanced_accuracy']:.3f}"
              f" | best so far {best_so_far:.3f}", log=not individualized, show=_SHOW_EPOCHS)
        return row["inner_balanced_accuracy"]

    sampler = optuna.samplers.TPESampler(
        seed=stable_seed("optuna", split.unit),
        n_startup_trials=min(5, max(1, budget // 2)),
    )
    study = optuna.create_study(direction="maximize", sampler=sampler)
    study.optimize(objective, n_trials=budget)
    best = candidates[study.best_trial.number]
    _live(f"  {split.unit} selected trial {best['candidate']}/{budget}"
          f" | refitting for {best['refit_epochs']} epochs", show=_SHOW_EPOCHS)
    return best, candidates


def linear_shap(model, train_features, test_features):
    """Exact interventional linear SHAP in positive-class log-odds."""
    scaler, classifier = model.steps[0][1], model.steps[1][1]
    background = scaler.transform(train_features).mean(axis=0, dtype=np.float64)
    test_standardized = scaler.transform(test_features)
    return (test_standardized - background) * classifier.coef_[0]

In [ ]:
# Model checkpoints and held-out records

def checkpoint_path(split, model):
    return RUN_DIR / "checkpoints" / split.regime / split.unit / f"{model}.npz"


def frozen_model_path(split, model):
    return checkpoint_path(split, model).with_suffix(".pt" if model == "DA_TCN" else ".joblib")


def read_checkpoint(split, model):
    path = checkpoint_path(split, model)
    if not path.exists():
        return None
    try:
        with np.load(path, allow_pickle=False) as file:
            result = {key: file[key] for key in file.files}
        result["record"] = json.loads(result["record"].item())
        if (not np.array_equal(result["train_index"], split.train)
                or not np.array_equal(result["test_index"], split.test)):
            raise ValueError("checkpoint split indices differ from the current split plan")
        if result["p_correct"].shape != (len(split.test),) or not np.isfinite(result["p_correct"]).all():
            raise ValueError("invalid checkpoint probabilities")
        if split.regime == "generalized" and model in PRIMARY_MODELS[:2] and not frozen_model_path(split, model).exists():
            raise FileNotFoundError("frozen model is missing")
    except (OSError, ValueError, KeyError, EOFError, BadZipFile) as error:
        print(f"Will recompute {split.regime}/{split.unit}/{model}: {error}")
        return None
    return result


def run_model(split, model, y, angles, lr_inputs):
    existing = read_checkpoint(split, model)
    if existing is not None:
        return
    train, test = split.train, split.test
    seed = stable_seed("refit", split.unit, model)
    if model == "DA_TCN":
        best, search = tune_da(angles, y, split)
        params = best["params"]
        fitted, _, _ = train_da(angles[train], y[train], params, seed, best["refit_epochs"])
        p, alpha, beta = predict_da(fitted, angles[test])
        outputs = {"alpha": alpha, "beta": beta}
    else:
        x = lr_inputs[model]
        best, search = tune_lr(x, y, split)
        params = best["params"]
        fitted = make_lr(params["C"]).fit(x[train], y[train])
        p = fitted.predict_proba(x[test])[:, 1]
        outputs = {"abs_shap": np.abs(linear_shap(fitted, x[train], x[test]))
                   .reshape(len(test), -1, len(ANGLE_NAMES)).mean(axis=1).astype(np.float32)}
    if not np.isfinite(p).all() or any(not np.isfinite(v).all() for v in outputs.values()):
        raise FloatingPointError("Nonfinite probabilities or explanations from the fitted model")

    # The checkpoint is the completion marker and is written after the frozen model.
    if split.regime == "generalized" and model in PRIMARY_MODELS[:2]:
        with atomic_output(frozen_model_path(split, model)) as temp:
            if model == "DA_TCN":
                torch.save({k: v.detach().cpu() for k, v in fitted.state_dict().items()}, temp)
            else:
                joblib.dump(fitted, temp)
    record = {"selection": best, "search": search, "refit_seed": seed}
    save_npz(checkpoint_path(split, model), record=np.array(json.dumps(record, allow_nan=False)),
             train_index=train, test_index=test, p_correct=p, **outputs)


def run_regime(regime, plan, meta, angles, lr_inputs):
    global _SHOW_EPOCHS
    splits = plan[regime]
    if not splits:
        print(f"No eligible units in {regime}. See splits/exclusions.csv.")
        return None

    _SHOW_EPOCHS = regime != "individualized" or VERBOSE_INDIVIDUALIZED
    hardware = torch.cuda.get_device_name(DEVICE) if DEVICE.type == "cuda" else "CPU"
    _live(f"{regime} | {len(splits)} units | device: {hardware} | {RUN_DIR.name}", log=True)
    if DEVICE.type == "cpu" and not PREVIEW:
        _live("Full profile is running on CPU. Check the Colab runtime type before continuing.")

    y = meta.y.to_numpy()
    with tqdm(total=len(splits) * len(PRIMARY_MODELS), desc=regime, unit="model") as bar:
        for i, split in enumerate(splits, 1):
            for model in PRIMARY_MODELS:
                label = f"{i}/{len(splits)} {split.unit} | {model}"
                bar.set_postfix_str(label, refresh=True)
                if read_checkpoint(split, model) is not None:
                    _live(f"{label} | reused completed fit", show=_SHOW_EPOCHS)
                else:
                    started = perf_counter()
                    _live(f"{label} | starting (train={len(split.train)}, test={len(split.test)})",
                          log=True, show=_SHOW_EPOCHS)
                    try:
                        run_model(split, model, y, angles, lr_inputs)
                    except Exception as error:
                        _live(f"{label} | failed: {error}", log=True)
                        raise RuntimeError(f"{label} failed. Rerun this section to resume.") from error
                    _live(f"{label} | saved | {perf_counter() - started:.0f}s",
                          log=True, show=_SHOW_EPOCHS)
                bar.update(1)

    _live(f"{regime} | exporting results", log=True)
    return export_regime(regime, splits, meta)


def prediction_rows(meta, split, model, p):
    frame = meta.iloc[split.test][["trial_id", "filename", "subject", "session", "exercise", "repetition", "posture", "y"]].copy()
    frame = frame.rename(columns={"y": "y_true"}).reset_index(drop=True)
    return frame.assign(regime=split.regime, unit=split.unit, fold=split.fold,
                        repeat=split.repeat, model=model, p_correct=p,
                        y_hat=(p >= 0.5).astype(int))


def collect_results(splits, meta):
    metrics_rows, pred_frames, tuning_rows, choices, attention, shap_values = [], [], [], [], [], []
    for split in splits:
        for model in PRIMARY_MODELS:
            payload = read_checkpoint(split, model)
            if payload is None:
                raise RuntimeError(f"Missing checkpoint for {split.unit}/{model}")
            p = payload["p_correct"]
            predictions = prediction_rows(meta, split, model, p)
            pred_frames.append(predictions)
            best = payload["record"]["selection"]
            tags = {**split.keys(), "model": model}
            metrics_rows.append({**tags, **binary_metrics(predictions.y_true, p), "n_train": len(split.train)})
            choices.append({**tags, **best["params"], "inner_balanced_accuracy": best["inner_balanced_accuracy"],
                            "refit_epochs": best.get("refit_epochs"), "refit_seed": payload["record"]["refit_seed"]})
            for i, candidate in enumerate(payload["record"]["search"], 1):
                tuning_rows.append({**tags, "candidate": i, "selected": i == best["candidate"],
                                    **candidate["params"],
                                    "inner_balanced_accuracy": candidate["inner_balanced_accuracy"],
                                    "inner_fold_scores": json.dumps(candidate["inner_fold_scores"]),
                                    "inner_best_epochs": json.dumps(candidate.get("inner_best_epochs"))})
            if model == "DA_TCN":
                arrays = np.concatenate([payload["alpha"], payload["beta"]], axis=1)
                cols = [f"alpha_t{t:03d}" for t in range(T)] + [f"beta_{a}" for a in ANGLE_NAMES]
                attention.append(pd.concat([predictions, pd.DataFrame(arrays, columns=cols)], axis=1))
            else:
                shap_values.append(pd.concat([
                    predictions, pd.DataFrame(payload["abs_shap"], columns=ANGLE_NAMES)
                ], axis=1))
    return (pd.DataFrame(metrics_rows), pd.concat(pred_frames, ignore_index=True),
            pd.DataFrame(tuning_rows), pd.DataFrame(choices),
            pd.concat(attention, ignore_index=True), pd.concat(shap_values, ignore_index=True))

In [ ]:
# Metric tables, paired comparisons and manuscript exports

def paired_model_tables(units, pairs, regime):
    model_pairs = list(itertools.combinations(PRIMARY_MODELS, 2))
    if regime == "individualized":
        base = pairs
        keys = ["subject", "exercise"]
        primary_unit = "subject_exercise_pair"
    else:
        base = units
        keys = ["unit", "exercise"] if regime == "per_exercise" else ["unit"]
        primary_unit = "outer_fold"
    wide = base.pivot(index=keys, columns="model", values="balanced_accuracy").reset_index()
    by_pair = pairs.pivot(index=["subject", "exercise"], columns="model",
                          values="balanced_accuracy").reset_index()
    deltas, tests, across, participant_rows = [], [], [], []
    for a, b in model_pairs:
        comparison = f"{a} - {b}"
        d = wide.dropna(subset=[a, b]).copy()
        d["delta"] = d[a] - d[b]
        deltas.append(d[keys + ["delta"]].assign(comparison=comparison, model_a=a, model_b=b))
        exercise_groups = d.groupby("exercise") if regime == "per_exercise" else [(None, d)]
        for exercise, sub in exercise_groups:
            groups = sub["subject"] if regime == "individualized" else None
            tests.append({"comparison": comparison, "model_a": a, "model_b": b, "exercise": exercise,
                          "evaluation_unit": primary_unit, **delta_summary(sub.delta, groups=groups)})
        if regime == "per_exercise":
            medians = d.groupby("exercise").delta.median()
            across.append({"comparison": comparison, "evaluation_unit": "exercise_median_fold_delta",
                           **delta_summary(medians)})
        d_subject = (by_pair.assign(delta=by_pair[a] - by_pair[b]).dropna(subset=["delta"])
                     .groupby("subject").delta.mean())
        lo, hi = bootstrap_median(d_subject)
        participant_rows.append({"comparison": comparison, "n_subjects": len(d_subject),
                                 "mean_subject_delta": d_subject.mean(),
                                 "median_subject_delta": d_subject.median(),
                                 "median_subject_ci_low": lo, "median_subject_ci_high": hi,
                                 "subject_sign_flip_p_exploratory": exact_sign_flip_p(d_subject)})
    comparison_table = pd.DataFrame(tests)
    if regime == "per_exercise":
        comparison_table["p_fdr_bh_across_exercises"] = (
            comparison_table.groupby("comparison")["wilcoxon_p_exploratory"].transform(bh_adjust)
        )
    return pd.concat(deltas, ignore_index=True), comparison_table, pd.DataFrame(across), pd.DataFrame(participant_rows)


def score_subject_exercise(pred, units, regime):
    keys = ["model", "subject", "exercise"]
    if regime == "individualized":
        return (units.groupby(keys, as_index=False)[list(METRICS)]
                .mean().merge(units.groupby(keys).size().rename("n_units"), on=keys))
    rows = []
    for (model, subject, exercise), group in pred.groupby(keys):
        rows.append({"model": model, "subject": subject, "exercise": exercise, "n_units": 1,
                     **binary_metrics(group.y_true, group.p_correct)})
    return pd.DataFrame(rows)


def export_regime(regime, splits, meta):
    folder = RUN_DIR / regime
    units, pred, tuning, choices, attention, shap = collect_results(splits, meta)
    n_expected = len(splits) * len(PRIMARY_MODELS)
    if len(units) != n_expected or units.duplicated(["unit", "model"]).any():
        raise RuntimeError(f"Expected {n_expected} unique model/unit results in {regime}")
    if pred.duplicated(["unit", "model", "trial_id"]).any():
        raise RuntimeError(f"Duplicate held-out predictions in {regime}")
    pairs = score_subject_exercise(pred, units, regime)
    base = pairs if regime == "individualized" else units
    overall = group_summary(base, ["model"], METRICS)
    overall.insert(0, "regime", regime)
    overall.insert(1, "evaluation_unit", "subject_exercise_pair" if regime == "individualized" else "outer_fold")
    by_exercise_base = pairs if regime == "individualized" else units
    by_exercise = group_summary(by_exercise_base, ["exercise", "model"], METRICS)
    deltas, comparisons, across, subjects = paired_model_tables(units, pairs, regime)

    tables = {
        "unit_metrics": units, "predictions": pred, "tuning_candidates": tuning,
        "selected_parameters": choices, "metric_summary": overall, "subject_exercise_metrics": pairs,
        "paired_deltas_balanced_accuracy": deltas, "paired_comparisons_balanced_accuracy": comparisons,
        "participant_sensitivity": subjects,
    }
    if regime != "generalized":
        tables["metric_summary_by_exercise"] = by_exercise
    if regime == "individualized":
        tables["subject_exercise_repeat_sd"] = (
            units.groupby(["model", "subject", "exercise"], as_index=False)[list(METRICS)].std()
        )
    if not across.empty:
        tables["across_exercise_comparison"] = across
    for name, frame in tables.items():
        save_csv(frame, folder / f"{name}.csv")
    export_explanations(regime, folder, attention, shap)
    if regime == "generalized":
        plot_classification(pred, folder / "figures")
    display(overall.pivot(index="model", columns="metric", values="mean").round(3))
    return {"units": units, "pred": pred, "pairs": pairs, "summary": overall,
            "comparisons": comparisons, "parameters": choices}


def export_manuscript_tables(results):
    folder = RUN_DIR / "manuscript"
    if not results:
        print("Run at least one primary regime to generate the manuscript tables.")
        return
    save_csv(pd.concat([r["summary"] for r in results.values()], ignore_index=True),
             folder / "metric_summary_all_regimes.csv")
    save_csv(pd.concat([r["units"] for r in results.values()], ignore_index=True),
             folder / "metrics_by_evaluation_unit.csv")
    save_csv(pd.concat([r["comparisons"].assign(regime=k) for k, r in results.items()], ignore_index=True),
             folder / "paired_comparisons_all_regimes.csv")
    selected = pd.concat([r["parameters"] for r in results.values()], ignore_index=True)
    save_csv(selected, folder / "selected_parameters_all_regimes.csv")
    da = selected[selected.model == "DA_TCN"]
    reports = []
    for regime, rows in da.groupby("regime", sort=False):
        record = {"regime": regime, "n_units": len(rows)}
        for col in ("hidden", "levels", "kernel", "batch"):
            record[col] = rows[col].dropna().mode().iloc[0]
        for col in ("dropout", "lr", "wd", "refit_epochs"):
            record[col] = rows[col].median()
        reports.append(record)
    save_csv(pd.DataFrame(reports), folder / "da_configuration_mode_median.csv")

    if all(regime in results for regime in REGIMES):
        keys = ["model", "subject", "exercise"]
        comparable = []
        for regime in REGIMES:
            p = results[regime]["pairs"][keys + ["balanced_accuracy"]].copy()
            comparable.append(p.rename(columns={"balanced_accuracy": regime}).dropna())
        matched = comparable[0].merge(comparable[1], on=keys).merge(comparable[2], on=keys)
        matched["individualized_minus_per_exercise"] = matched.individualized - matched.per_exercise
        matched["per_exercise_minus_generalized"] = matched.per_exercise - matched.generalized
        save_csv(matched, folder / "matched_pairs_across_regimes_descriptive.csv")
        if len(matched):
            save_csv(group_summary(matched, ["model"], REGIMES), folder / "matched_pairs_across_regimes_summary.csv")
    print(f"Manuscript tables: {folder}")

In [ ]:
# Held-out attention, linear SHAP and figures

def class_contrast(frame, keys, measures):
    """Long table of class means and correct-minus-incorrect contrasts."""
    grouped = frame.groupby([*keys, "y_true"], dropna=False)[list(measures)].mean().reset_index()
    long = grouped.melt(id_vars=[*keys, "y_true"], var_name="feature", value_name="mean")
    wide = long.pivot(index=[*keys, "feature"], columns="y_true", values="mean")
    wide = wide.reindex(columns=[0, 1]).rename(columns={0: "mean_incorrect", 1: "mean_correct"})
    wide["delta_correct_minus_incorrect"] = wide.mean_correct - wide.mean_incorrect
    return wide.rename_axis(columns=None).reset_index()


def class_lines(ax, data, x, columns):
    for label, frame in data.groupby("y_true"):
        ax.plot(x, frame[columns].mean(), color=LABEL_COLORS[label],
                label=f"{'Correct' if label else 'Incorrect'} (n={len(frame)})")
    ax.legend(frameon=False)


def export_explanations(regime, folder, attention, shap):
    explain = folder / "explanations"
    alpha_cols = [f"alpha_t{t:03d}" for t in range(T)]
    beta_cols = [f"beta_{angle}" for angle in ANGLE_NAMES]
    alpha, beta = attention[alpha_cols].to_numpy(), attention[beta_cols].to_numpy()
    phase_names = [f"phase_{i + 1}" for i in range(SETTINGS["phase_bins"])]
    for name, block in zip(phase_names, np.array_split(alpha, SETTINGS["phase_bins"], axis=1)):
        attention[name] = block.sum(axis=1)
    for name, arr in (("entropy_alpha", alpha), ("entropy_beta", beta)):
        attention[name] = -np.sum(arr * np.log(np.clip(arr, 1e-12, 1.0)), axis=1)
    rank_cols = ["top_angle_1", "top_angle_2", "top_angle_3"]
    attention[rank_cols] = np.asarray(ANGLE_NAMES)[np.argsort(-beta, axis=1, kind="stable")[:, :3]]
    save_csv(attention, explain / "attention_heldout.csv")
    measures = [*alpha_cols, *beta_cols, *phase_names, "entropy_alpha", "entropy_beta"]
    # A trial appearing in several individualized tests receives one weight in pooled plots.
    trial_attention = (attention.groupby(["trial_id", "subject", "exercise", "y_true"], as_index=False)[measures]
                       .mean())
    for name, columns in (("beta", beta_cols), ("phases", phase_names),
                          ("entropy", ["entropy_alpha", "entropy_beta"])):
        save_csv(class_contrast(trial_attention, [], columns), explain / f"{name}_by_class.csv")
        save_csv(class_contrast(trial_attention, ["exercise"], columns), explain / f"{name}_by_exercise.csv")
        if regime == "individualized":
            pair = class_contrast(trial_attention, ["subject", "exercise"], columns)
            save_csv(pair, explain / f"{name}_by_subject_exercise.csv")
            pair_summary = (pair.groupby(["exercise", "feature"])["delta_correct_minus_incorrect"]
                            .agg(n_pairs="count", mean_delta="mean", sd_delta="std").reset_index())
            save_csv(pair_summary, explain / f"{name}_pair_contrast_by_exercise.csv")
    failures = attention.loc[attention.y_true == 0]
    ranks = failures.melt(id_vars=["unit", "exercise"], value_vars=rank_cols,
                         var_name="rank", value_name="angle")
    frequencies = ranks.groupby(["unit", "exercise", "angle"]).size().rename("count").reset_index()
    frequencies["fraction_of_top3_slots"] = (
        frequencies["count"] / frequencies.groupby(["unit", "exercise"])["count"].transform("sum")
    )
    save_csv(frequencies, explain / "failure_top3_by_unit.csv")
    dominant = (frequencies.sort_values(["unit", "exercise", "count", "angle"],
                                        ascending=[True, True, False, True])
                .drop_duplicates(["unit", "exercise"]))
    save_csv(dominant, explain / "failure_dominant_angle_by_unit.csv")
    consistency = dominant.groupby(["exercise", "angle"]).size().rename("n_units").reset_index()
    consistency["fraction_units"] = consistency.n_units / consistency.groupby("exercise").n_units.transform("sum")
    save_csv(consistency, explain / "failure_dominant_angle_consistency.csv")
    trial_beta = trial_attention.loc[trial_attention.y_true == 0]
    signatures = (trial_beta[["exercise"]].assign(
        angle=np.asarray(ANGLE_NAMES)[trial_beta[beta_cols].to_numpy().argmax(axis=1)])
        .groupby(["exercise", "angle"]).size().rename("n_trials").reset_index())
    signatures["fraction"] = signatures.n_trials / signatures.groupby("exercise").n_trials.transform("sum")
    save_csv(signatures, explain / "failure_top_angle_by_exercise.csv")
    if regime == "individualized":
        stability = attention.groupby(["subject", "exercise", "repeat"])[beta_cols].mean().reset_index()
        stability["angle"] = np.asarray(ANGLE_NAMES)[stability[beta_cols].to_numpy().argmax(axis=1)]
        save_csv(stability[["subject", "exercise", "repeat", "angle"]],
                 explain / "dominant_angle_by_repeat.csv")
        frequencies = stability.groupby(["subject", "exercise", "angle"]).size().rename("n_repeats").reset_index()
        frequencies["fraction_repeats"] = (
            frequencies.n_repeats / frequencies.groupby(["subject", "exercise"]).n_repeats.transform("sum")
        )
        save_csv(frequencies, explain / "dominant_angle_stability_by_pair.csv")

    shap_long = shap.melt(id_vars=["model", "unit", "fold", "repeat", "subject", "exercise", "trial_id", "y_true"],
                          value_vars=ANGLE_NAMES, var_name="angle", value_name="mean_abs_shap")
    save_csv(shap_long, explain / "shap_heldout_by_angle.csv")
    unit_shap = (shap_long.groupby(["model", "unit", "exercise", "angle"])["mean_abs_shap"]
                 .agg(n_test="count", mean_abs_shap="mean").reset_index())
    save_csv(unit_shap, explain / "shap_summary_by_unit_exercise.csv")
    if regime == "individualized":
        per_trial = shap_long.groupby(["model", "trial_id", "subject", "exercise", "angle"], as_index=False).mean_abs_shap.mean()
        pair_shap = per_trial.groupby(["model", "subject", "exercise", "angle"], as_index=False).mean_abs_shap.mean()
        save_csv(pair_shap, explain / "shap_summary_by_subject_exercise.csv")
        base = pair_shap
        save_csv(pair_shap.sort_values("mean_abs_shap", ascending=False)
                 .groupby(["model", "subject", "exercise"]).head(3),
                 explain / "shap_top3_by_subject_exercise.csv")
    else:
        base = unit_shap
    shap_summary = (base.groupby(["model", "exercise", "angle"]).mean_abs_shap
                    .agg(n_units="count", mean="mean", sd="std").reset_index())
    save_csv(shap_summary, explain / "shap_summary_by_exercise.csv")

    figs = folder / "figures"
    fig, ax = plt.subplots(figsize=(7, 3.5))
    class_lines(ax, trial_attention, np.linspace(0, 100, T), alpha_cols)
    ax.set(xlabel="Normalized repetition (%)", ylabel="Mean temporal attention (alpha)", title="Held-out temporal attention")
    save_figure(fig, figs / "attention_temporal_by_class")
    fig, ax = plt.subplots(figsize=(7, 4.2))
    for label, frame in trial_attention.groupby("y_true"):
        ax.barh(np.arange(len(beta_cols)) + (0.18 if label else -0.18), frame[beta_cols].mean(),
                height=0.34, color=LABEL_COLORS[label], label="Correct" if label else "Incorrect")
    ax.set(yticks=np.arange(len(beta_cols)), yticklabels=ANGLE_NAMES, xlabel="Mean angle attention (beta)")
    ax.invert_yaxis()
    ax.legend(frameon=False)
    save_figure(fig, figs / "attention_angle_by_class")

    if regime == "generalized":
        for label, data in attention.groupby("y_true"):
            # Select a held-out illustration by confidence in the true label.
            confidence = data.p_correct if label else 1 - data.p_correct
            row = data.loc[confidence.idxmax()]
            fig, (ax, bx) = plt.subplots(2, 1, figsize=(7, 6))
            ax.plot(np.linspace(0, 100, T), row[alpha_cols].to_numpy(float), color=LABEL_COLORS[label])
            ax.set(xlabel="Normalized repetition (%)", ylabel="Alpha")
            bx.barh(ANGLE_NAMES, row[beta_cols].to_numpy(float), color=LABEL_COLORS[label])
            bx.invert_yaxis()
            bx.set(xlabel="Beta")
            fig.suptitle(f"Held-out {row.trial_id} | true label {label} | P(correct) {row.p_correct:.2f}")
            fig.tight_layout()
            save_figure(fig, figs / f"attention_example_{'correct' if label else 'incorrect'}")


def plot_classification(pred, folder):
    fig, ax = plt.subplots(figsize=(5.5, 4.5))
    for model, group in pred.groupby("model", sort=False):
        if group.y_true.nunique() != 2:
            continue
        x, y, _ = roc_curve(group.y_true, group.p_correct)
        auc = roc_auc_score(group.y_true, group.p_correct)
        ax.plot(x, y, label=f"{MODEL_LABELS[model]} (AUC {auc:.3f})", color=COLORS[model])
    ax.plot([0, 1], [0, 1], "--", color="#999999", linewidth=1)
    ax.set(xlabel="False positive rate", ylabel="True positive rate", title="Pooled held-out ROC")
    ax.legend(frameon=False)
    save_figure(fig, folder / "roc_oof")
    fig, axes = plt.subplots(1, len(PRIMARY_MODELS), figsize=(10.5, 3.5), constrained_layout=True)
    for ax, model in zip(axes, PRIMARY_MODELS):
        group = pred[pred.model == model]
        cm = confusion_matrix(group.y_true, group.y_hat, labels=[0, 1])
        ax.imshow(cm, cmap="Blues")
        for i, j in itertools.product(range(2), repeat=2):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                    color="white" if cm[i, j] > 0.6 * cm.max() else "black")
        ax.set(xticks=[0, 1], yticks=[0, 1], xlabel="Predicted", ylabel="True", title=MODEL_LABELS[model])
    save_figure(fig, folder / "confusion_counts_oof")

## 1. Generalized

Fit on all gestures from the training participants and evaluate on unseen participants. The five grouped outer folds share their test sets across models. Outputs include fold scores, out-of-fold predictions, ROC and confusion figures, model comparisons, held-out attention, and SHAP.

In [ ]:
results["generalized"] = run_regime("generalized", plan, meta, X, lr_inputs)

generalized | 5 units | device: NVIDIA L4 | intellirehabds_three_regime_v2_gpu


generalized:   0%|          | 0/15 [00:00<?, ?model/s]

1/5 fold01 | LR_angles | reused completed fit
1/5 fold01 | DA_TCN | reused completed fit
1/5 fold01 | LR_features | reused completed fit
2/5 fold02 | LR_angles | reused completed fit
2/5 fold02 | DA_TCN | reused completed fit
2/5 fold02 | LR_features | reused completed fit
3/5 fold03 | LR_angles | reused completed fit
3/5 fold03 | DA_TCN | reused completed fit
3/5 fold03 | LR_features | reused completed fit
4/5 fold04 | LR_angles | reused completed fit
4/5 fold04 | DA_TCN | reused completed fit
4/5 fold04 | LR_features | reused completed fit
5/5 fold05 | LR_angles | reused completed fit
5/5 fold05 | DA_TCN | reused completed fit
5/5 fold05 | LR_features | reused completed fit
generalized | exporting results


metric,accuracy,auc,balanced_accuracy,f1,precision,sensitivity,specificity
model,,,,,,,
DA_TCN,0.734,0.777,0.702,0.815,0.895,0.757,0.646
LR_angles,0.808,0.878,0.783,0.872,0.921,0.832,0.734
LR_features,0.736,0.732,0.654,0.817,0.892,0.754,0.554


### Movement Quality Metrics and biomechanical validation

This section uses the original six keyframes and computes Pose Similarity Measure (PSM), Motion Smoothness Index (MSI), and Joint Deviation (JD) from the three-dimensional coordinates. Templates use correct executions from the current outer training participants. Held-out comparisons and effects are exported by gesture and overall. The biomechanics tables preserve the joint-angle order and use normalized time.

In [ ]:
# Reference templates, movement quality descriptors and biomechanical effects

def keyframes(sequence, k=6):
    energy = np.linalg.norm(np.diff(sequence, axis=0).reshape(len(sequence) - 1, -1), axis=1)
    peaks = np.argsort(energy)[::-1][:k - 2] + 1 if energy.sum() else []
    indices = np.unique(np.asarray([0, *peaks, len(sequence) - 1], dtype=int))
    return resample_frames(sequence[indices], k)


def mqm_values(rep, template):
    dot = np.sum(rep * template, axis=-1)
    denom = np.linalg.norm(rep, axis=-1) * np.linalg.norm(template, axis=-1)
    return {"PSM": float(np.mean(dot / (denom + 1e-9))),
            "MSI": float(np.mean(np.linalg.norm(np.diff(rep, n=2, axis=0), axis=-1))),
            "JD": float(np.mean(np.linalg.norm(rep - template, axis=-1)))}


def two_class_effect(correct, incorrect):
    x, y = np.asarray(correct, float), np.asarray(incorrect, float)
    if not len(x) or not len(y):
        return {"n_correct": len(x), "n_incorrect": len(y), "cohens_d": np.nan,
                "cliffs_delta": np.nan, "mann_whitney_p_exploratory": np.nan}
    u, p = mannwhitneyu(x, y, alternative="two-sided")
    dof = len(x) + len(y) - 2
    ss = np.square(x - x.mean()).sum() + np.square(y - y.mean()).sum()
    denom = np.sqrt(ss / dof) if dof > 0 else 0.0
    return {"n_correct": len(x), "n_incorrect": len(y), "cohens_d": (x.mean() - y.mean()) / denom if denom else np.nan,
            "cliffs_delta": 2 * u / (len(x) * len(y)) - 1, "mann_whitney_p_exploratory": p}


def descriptor_statistics(frame, measures):
    rows = []
    for exercise, group in [("all", frame), *list(frame.groupby("exercise", sort=True))]:
        for measure in measures:
            correct = group.loc[group.y_true == 1, measure]
            incorrect = group.loc[group.y_true == 0, measure]
            means = group.groupby(["subject", "exercise", "y_true"])[measure].mean().unstack("y_true").reindex(columns=[0, 1])
            subject_diff = (means[1] - means[0]).dropna().groupby("subject").mean()
            rows.append({
                "exercise": exercise, "metric": measure, "mean_correct": correct.mean(),
                "mean_incorrect": incorrect.mean(), **two_class_effect(correct, incorrect),
                "n_paired_subjects": len(subject_diff), "median_subject_delta": subject_diff.median(),
                "subject_sign_flip_p_exploratory": exact_sign_flip_p(subject_diff),
            })
    table = pd.DataFrame(rows)
    table["p_fdr_bh_exploratory"] = table.groupby("exercise")["mann_whitney_p_exploratory"].transform(bh_adjust)
    return table


def run_mqm(meta, coco, splits):
    folder = RUN_DIR / "generalized" / "mqm"
    keyframe_cache = np.stack([keyframes(sequence) for sequence in coco])
    meta_columns = ["trial_id", "subject", "exercise", "y"]
    rows, reference_audit = [], []
    for split in splits:
        training = meta.iloc[split.train]
        held_out = meta.iloc[split.test]
        for exercise, test in held_out.groupby("exercise", sort=True):
            correct = training[(training.exercise == exercise) & (training.y == 1)]
            reference_audit.append({"fold": split.fold, "exercise": exercise,
                                    "n_correct_training": len(correct), "n_held_out": len(test),
                                    "has_reference": len(correct) >= 3})
            if len(correct) < 3:
                continue
            template = keyframes(coco[correct.index.to_numpy()].mean(axis=0))
            for i, row in test[meta_columns].iterrows():
                rows.append({"fold": split.fold, "trial_id": row.trial_id, "subject": row.subject,
                             "exercise": exercise, "y_true": row.y,
                             **mqm_values(keyframe_cache[i], template)})
    save_csv(pd.DataFrame(reference_audit), folder / "reference_audit.csv")
    if not rows:
        print("MQM requires at least three correct training trials for an exercise.")
        return
    data = pd.DataFrame(rows)
    measures = ["PSM", "MSI", "JD"]
    save_csv(data, folder / "mqm_heldout_repetitions.csv")
    save_csv(group_summary(data, ["exercise", "y_true"], measures), folder / "mqm_by_exercise_class.csv")
    save_csv(group_summary(data, ["y_true"], measures), folder / "mqm_by_class.csv")
    save_csv(descriptor_statistics(data, measures), folder / "mqm_class_comparisons.csv")
    print(f"MQM descriptors from {len(data)} held-out repetitions.")


def run_biomechanical_validation(meta, angles):
    folder = RUN_DIR / "generalized" / "explanations"
    stats = {
        "rom": np.ptp(angles, axis=1),
        "mean_abs_d1": np.abs(np.diff(angles, axis=1)).mean(axis=1),
        "mean_abs_d3": np.abs(np.diff(angles, n=3, axis=1)).mean(axis=1),
    }
    frame = meta[["trial_id", "subject", "exercise", "y"]].rename(columns={"y": "y_true"}).copy()
    measures = []
    for stat, values in stats.items():
        for i, angle in enumerate(ANGLE_NAMES):
            name = f"{stat}_{angle}"
            frame[name] = values[:, i]
            measures.append(name)
    save_csv(frame, folder / "biomechanical_descriptors.csv")
    effects = descriptor_statistics(frame, measures)
    save_csv(effects, folder / "biomechanical_class_effects.csv")
    attention = pd.read_csv(folder / "attention_heldout.csv")
    joined = attention[["trial_id", *[f"beta_{a}" for a in ANGLE_NAMES]]].merge(
        frame, on="trial_id", validate="one_to_one")
    save_csv(joined, folder / "attention_and_biomechanical_descriptors.csv")
    beta = pd.read_csv(folder / "beta_by_class.csv").set_index("feature")
    aggregate = effects[effects.exercise == "all"].set_index("metric")
    rows = []
    for angle in ANGLE_NAMES:
        row = beta.loc[f"beta_{angle}"]
        rows.append({"angle": angle, "mean_beta": attention[f"beta_{angle}"].mean(),
                     "beta_correct": row.mean_correct, "beta_incorrect": row.mean_incorrect,
                     "abs_beta_contrast": abs(row.delta_correct_minus_incorrect),
                     "abs_d_rom": abs(aggregate.loc[f"rom_{angle}", "cohens_d"]),
                     "abs_d3_effect": abs(aggregate.loc[f"mean_abs_d3_{angle}", "cohens_d"])})
    alignment = pd.DataFrame(rows)
    save_csv(alignment, folder / "attention_effect_alignment.csv")
    fig, ax = plt.subplots(figsize=(6.5, 4.5))
    focus = alignment[alignment.angle.isin(["L_elbow", "R_elbow", "L_shoulder", "R_shoulder"])]
    ax.scatter(focus.mean_beta, focus.abs_d3_effect, color=COLORS["DA_TCN"])
    for row in focus.itertuples():
        ax.annotate(row.angle, (row.mean_beta, row.abs_d3_effect), xytext=(3, 4), textcoords="offset points")
    ax.set(xlabel="Mean held-out angle attention", ylabel="Absolute Cohen’s d (third difference)",
           title="Attention and biomechanical effect sizes")
    save_figure(fig, RUN_DIR / "generalized" / "figures" / "attention_effect_alignment")

In [ ]:
run_mqm(meta, coco, plan["generalized"])
if results["generalized"] is not None:
    run_biomechanical_validation(meta, X)

MQM descriptors from 2577 held-out repetitions.


### Skeleton corruption robustness

Apply spatial jitter, random joint dropout, burst occlusion, and frame drop to held-out 3D skeletons. Evaluate the frozen generalized LR and DA-TCN on the same perturbed recordings. The notebook exports fold and seed scores, slopes, the LR–DA-TCN gap, and plots.

In [ ]:
# Test-time corruptions, frozen-model evaluation and slopes

CORRUPTION_LEVELS = {
    "jitter": ([0.0, 0.01, 0.02, 0.05], 0.10),
    "dropout": ([0.05, 0.10, 0.20], 0.40),
    "burst": ([0.10, 0.25, 0.40], 0.60),
    "framedrop": ([0.05, 0.10, 0.20], 0.40),
}
CORRUPTION_LABELS = {"jitter": "Spatial jitter", "dropout": "Joint dropout",
                     "burst": "Burst occlusion", "framedrop": "Frame drop"}


def interpolate_mask(sequence, mask):
    out = sequence.copy()
    times = np.arange(len(sequence))
    for joint in np.flatnonzero(mask.any(axis=0)):
        good = ~mask[:, joint]
        if not good.any():
            out[:, joint] = 0
            continue
        for coordinate in range(out.shape[-1]):
            out[~good, joint, coordinate] = np.interp(
                times[~good], times[good], sequence[good, joint, coordinate],
            )
    return out


def corrupt_sequence(sequence, family, severity, rng):
    if severity == 0:
        return sequence
    n_frames, n_joints, _ = sequence.shape
    if family == "jitter":
        scale = np.median(np.linalg.norm(sequence[:, 5] - sequence[:, 6], axis=-1))
        if scale < 1e-8:
            scale = max(float(sequence.std()), 1e-6)
        return (sequence + rng.normal(0, severity * scale, sequence.shape)).astype(np.float32)
    if family == "framedrop":
        n_drop = min(n_frames - 2, max(1, round(severity * n_frames)))
        keep = np.sort(rng.choice(n_frames, size=n_frames - n_drop, replace=False))
        return resample_frames(sequence[keep], T)
    if family == "dropout":
        mask = rng.random((n_frames, n_joints)) < severity
    elif family == "burst":
        mask = np.zeros((n_frames, n_joints), dtype=bool)
        width = min(n_frames, max(1, round(severity * n_frames)))
        joint = rng.choice([7, 8, 9, 10, 13, 14, 15, 16])
        start = rng.integers(0, n_frames - width + 1)
        mask[start:start + width, joint] = True
    else:
        raise ValueError(f"Unknown corruption family {family}")
    return interpolate_mask(sequence, mask)


def reload_generalized_models(splits):
    frozen = {}
    for split in splits:
        payloads = {model: read_checkpoint(split, model) for model in PRIMARY_MODELS[:2]}
        if any(value is None for value in payloads.values()):
            raise RuntimeError(f"Complete the generalized section before robustness ({split.unit} is missing).")
        lr = joblib.load(frozen_model_path(split, "LR_angles"))
        da = DualAttentionTCN(payloads["DA_TCN"]["record"]["selection"]["params"]).to(DEVICE)
        da.load_state_dict(torch.load(frozen_model_path(split, "DA_TCN"), map_location=DEVICE, weights_only=True))
        frozen[split.unit] = (lr, da)
    return frozen


def robustness_slopes(seed_means):
    rows = []
    for family, subset in seed_means.groupby("family", sort=False):
        for severity_range in ("standard", "including_extreme"):
            data = subset.loc[~subset.extreme] if severity_range == "standard" else subset
            for model, part in data.groupby("model", sort=False):
                wide = part.pivot(index="seed_rep", columns="severity", values="balanced_accuracy").dropna()
                levels = wide.columns.to_numpy(dtype=float)
                if len(wide) == 0 or len(levels) < 2:
                    continue
                values = wide.to_numpy()
                slopes = np.polyfit(levels, values.T, 1)[0]
                lo, hi = np.nan, np.nan
                if len(slopes) >= 2:
                    rng = np.random.default_rng(stable_seed("robustness_ci", family, severity_range))
                    boot = rng.choice(slopes, size=(SETTINGS["robustness_bootstrap"], len(slopes)),
                                      replace=True).mean(axis=1)
                    lo, hi = np.quantile(boot, (0.025, 0.975))
                rows.append({"family": family, "severity_range": severity_range, "model": model,
                             "n_seeds": len(wide), "n_levels": len(levels), "slope": slopes.mean(),
                             "ci_low": lo, "ci_high": hi, "ba_at_min": values[:, 0].mean(),
                             "ba_at_max": values[:, -1].mean(),
                             "delta_max_minus_min": (values[:, -1] - values[:, 0]).mean()})
    return pd.DataFrame(rows)


def run_robustness(meta, angles, coco, splits):
    if not splits:
        return
    folder = RUN_DIR / "generalized" / "robustness"
    frozen = reload_generalized_models(splits)
    y = meta.y.to_numpy()
    tested = np.unique(np.concatenate([split.test for split in splits]))
    conditions = [(family, level, level == high) for family, (standard, high) in CORRUPTION_LEVELS.items()
                  for level in [*standard, high]]
    frames = []
    for family, level, extreme in tqdm(conditions, desc="Skeleton corruptions"):
        for rep in range(1, SETTINGS["robustness_seeds"] + 1):
            path = folder / "checkpoints" / f"{family}_{level:.3f}_seed{rep:02d}.csv"
            if path.exists():
                frames.append(pd.read_csv(path))
                continue
            seed = stable_seed("robustness", family, level, rep)
            rng = np.random.default_rng(seed)
            altered = angles.copy()
            if level:
                altered[tested] = np.stack([compute_angles(corrupt_sequence(coco[i], family, level, rng))
                                            for i in tested])
            rows = []
            for split in splits:
                x = altered[split.test]
                lr, da = frozen[split.unit]
                p_lr = lr.predict_proba(x.reshape(len(x), -1))[:, 1]
                p_da, _, _ = predict_da(da, x)
                for model, p in zip(PRIMARY_MODELS[:2], (p_lr, p_da)):
                    rows.append({"family": family, "severity": level, "extreme": extreme,
                                 "seed_rep": rep, "seed": seed, "unit": split.unit, "model": model,
                                 **binary_metrics(y[split.test], p)})
            frame = pd.DataFrame(rows)
            save_csv(frame, path)
            frames.append(frame)

    fold_scores = pd.concat(frames, ignore_index=True)
    keys = ["family", "severity", "extreme", "seed_rep"]
    seed_means = fold_scores.groupby([*keys, "model"], as_index=False)[list(METRICS)].mean()
    summary = group_summary(seed_means, ["family", "severity", "extreme", "model"], METRICS)
    gap = seed_means.pivot(index=keys, columns="model", values="balanced_accuracy").reset_index()
    gap["LR_minus_DA"] = gap["LR_angles"] - gap["DA_TCN"]
    gap_summary = group_summary(gap, ["family", "severity", "extreme"], ["LR_minus_DA"])
    slopes = robustness_slopes(seed_means)
    slope_gap = slopes.pivot(index=["family", "severity_range"], columns="model", values="slope").reset_index()
    slope_gap["slope_LR_minus_DA"] = slope_gap["LR_angles"] - slope_gap["DA_TCN"]
    for name, table in {
        "fold_scores": fold_scores, "seed_means": seed_means, "severity_summary": summary,
        "model_gap_by_seed": gap, "model_gap_summary": gap_summary, "slopes": slopes, "slope_gap": slope_gap,
    }.items():
        save_csv(table, folder / f"{name}.csv")

    fig, axes = plt.subplots(2, 2, figsize=(9, 6), constrained_layout=True)
    for ax, (family, title) in zip(axes.ravel(), CORRUPTION_LABELS.items()):
        panel = summary[(summary.family == family) & (summary.metric == "balanced_accuracy")]
        for model, part in panel.groupby("model", sort=False):
            part = part.sort_values("severity")
            ax.errorbar(part.severity, part["mean"], yerr=part.sd.fillna(0), marker="o",
                        color=COLORS[model], label=MODEL_LABELS[model], capsize=3)
        ax.set(title=title, xlabel="Corruption severity", ylabel="Balanced accuracy")
    axes[0, 0].legend(frameon=False)
    save_figure(fig, folder / "figures" / "robustness_curves")
    display(slopes.round(4))

In [ ]:
run_robustness(meta, X, coco, plan["generalized"])

Skeleton corruptions:   0%|          | 0/17 [00:00<?, ?it/s]

,family,severity_range,model,n_seeds,n_levels,slope,ci_low,ci_high,ba_at_min,ba_at_max,delta_max_minus_min
0,burst,standard,DA_TCN,10,3,-0.0151,-0.0205,-0.0102,0.7014,0.6969,-0.0045
1,burst,standard,LR_angles,10,3,-0.0170,-0.0288,-0.0037,0.7815,0.7764,-0.0051
2,burst,including_extreme,DA_TCN,10,4,-0.0317,-0.0355,-0.0277,0.7014,0.6858,-0.0156
3,burst,including_extreme,LR_angles,10,4,-0.0146,-0.0215,-0.0076,0.7815,0.7755,-0.0060
4,dropout,standard,DA_TCN,10,3,0.0012,-0.0028,0.0057,0.7011,0.7013,0.0002
5,dropout,standard,LR_angles,10,3,-0.0106,-0.0233,0.0029,0.7811,0.7796,-0.0015
6,dropout,including_extreme,DA_TCN,10,4,0.0029,-0.0009,0.0073,0.7011,0.7022,0.0011
7,dropout,including_extreme,LR_angles,10,4,0.0122,0.0036,0.0202,0.7811,0.7853,0.0042
8,framedrop,standard,DA_TCN,10,3,0.0015,-0.0088,0.0107,0.7005,0.7009,0.0004
9,framedrop,standard,LR_angles,10,3,0.0397,0.0094,0.0707,0.7820,0.7879,0.0060


## 2. Per gesture

Fit and evaluate within each gesture while keeping participants disjoint. The original notebook recorded three one-class test folds. This version logs any one-class outer fold or invalid inner fold and uses the same eligible folds for all models. The preflight table shows the actual fold count by gesture. Outputs are under `per_exercise/`.

The figure cell saves one pooled overview and a separate set for each gesture. ROC curves use the eligible held-out predictions.

In [ ]:
results["per_exercise"] = run_regime("per_exercise", plan, meta, X, lr_inputs)

per_exercise | 42 units | device: NVIDIA L4 | intellirehabds_three_regime_v2_gpu


per_exercise:   0%|          | 0/126 [00:00<?, ?model/s]

1/42 ex00_fold01 | LR_angles | reused completed fit
1/42 ex00_fold01 | DA_TCN | reused completed fit
1/42 ex00_fold01 | LR_features | reused completed fit
2/42 ex00_fold02 | LR_angles | reused completed fit
2/42 ex00_fold02 | DA_TCN | reused completed fit
2/42 ex00_fold02 | LR_features | reused completed fit
3/42 ex00_fold03 | LR_angles | reused completed fit
3/42 ex00_fold03 | DA_TCN | reused completed fit
3/42 ex00_fold03 | LR_features | reused completed fit
4/42 ex00_fold04 | LR_angles | reused completed fit
4/42 ex00_fold04 | DA_TCN | reused completed fit
4/42 ex00_fold04 | LR_features | reused completed fit
5/42 ex00_fold05 | LR_angles | reused completed fit
5/42 ex00_fold05 | DA_TCN | reused completed fit
5/42 ex00_fold05 | LR_features | reused completed fit
6/42 ex01_fold01 | LR_angles | reused completed fit
6/42 ex01_fold01 | DA_TCN | reused completed fit
6/42 ex01_fold01 | LR_features | reused completed fit
7/42 ex01_fold02 | LR_angles | reused completed fit
7/42 ex01_fold02 |

metric,accuracy,auc,balanced_accuracy,f1,precision,sensitivity,specificity
model,,,,,,,
DA_TCN,0.800,0.742,0.662,0.838,0.860,0.856,0.468
LR_angles,0.857,0.871,0.821,0.895,0.947,0.862,0.781
LR_features,0.818,0.841,0.771,0.859,0.934,0.820,0.721


In [ ]:
if (RUN_DIR / "per_exercise" / "predictions.csv").exists():
    # Per-exercise ROC curves and confusion matrices
    per_exercise_dir = RUN_DIR / "per_exercise"
    per_exercise_pred = pd.read_csv(per_exercise_dir / "predictions.csv")
    figure_dir = per_exercise_dir / "figures"

    plot_classification(per_exercise_pred, figure_dir)

    for exercise, exercise_pred in per_exercise_pred.groupby("exercise", sort=True):
        plot_classification(
            exercise_pred,
            figure_dir / f"exercise_{int(exercise):02d}",
        )

    print("Saved per-exercise figures in", figure_dir)

Saved per-exercise figures in /content/drive/MyDrive/NeuroYOLO_IRDS (1)/clean_results/intellirehabds_three_regime_v2_gpu/per_exercise/figures


## 3. Individualized

Each participant–gesture pair needs at least two correct and two incorrect recordings on each side of the outer split. Holdouts target 30% per class, subject to those minima. Inner tuning uses three stratified folds when the training class counts support it and two for the sparsest eligible pairs. This keeps the minimum class rule used to identify the 16 matched pairs in the original notebook. The preflight reports the actual eligible cohort and exclusions.

The full profile draws 30 shared holdouts per pair. Metrics are averaged over repeats within each pair before the regime summary. Pair comparisons include participant-cluster bootstrap intervals and a participant-level sensitivity analysis. The ROC curves and row-normalized confusion matrices use the same equal weight per pair.

In [ ]:
results["individualized"] = run_regime("individualized", plan, meta, X, lr_inputs)

individualized | 480 units | device: NVIDIA L4 | intellirehabds_three_regime_v2_gpu


individualized:   0%|          | 0/1440 [00:00<?, ?model/s]

Streaming output truncated to the last 5000 lines.
    inner fit: starting epoch 5/40 | 0s elapsed | best validation BA 0.500
    inner fit: starting epoch 10/40 | 0s elapsed | best validation BA 0.500
    inner fit: starting epoch 15/40 | 0s elapsed | best validation BA 0.500
    inner fold 1 done | BA 0.500 | best epoch 13 | 0s
    inner fold 2/3 | train=5 | validation=3
    inner fit: starting epoch 1/40 | 0s elapsed
    inner fit: starting epoch 5/40 | 0s elapsed | best validation BA 0.500
    inner fit: starting epoch 10/40 | 0s elapsed | best validation BA 1.000
    inner fold 2 done | BA 1.000 | best epoch 6 | 0s
    inner fold 3/3 | train=6 | validation=2
    inner fit: starting epoch 1/40 | 0s elapsed
    inner fit: starting epoch 5/40 | 0s elapsed | best validation BA 0.500
    inner fit: starting epoch 10/40 | 0s elapsed | best validation BA 0.500
    inner fold 3 done | BA 0.500 | best epoch 2 | 0s
  DA-TCN s209_ex02_r09 | trial 10/10 done | mean BA 0.667 | best so far 1.00

In [ ]:
if (RUN_DIR / "individualized" / "predictions.csv").exists():
    # Individualized ROC curves and confusion matrices
    individual_dir = RUN_DIR / "individualized"
    individual_pred = pd.read_csv(individual_dir / "predictions.csv")
    fpr_grid = np.linspace(0, 1, 1001)

    roc_fig, roc_ax = plt.subplots(figsize=(5.5, 4.5))
    cm_fig, cm_axes = plt.subplots(
        1, len(PRIMARY_MODELS), figsize=(10.5, 3.5), constrained_layout=True
    )

    for model, cm_ax in zip(PRIMARY_MODELS, cm_axes):
        model_pred = individual_pred[individual_pred["model"] == model]
        pair_curves, pair_aucs, pair_matrices = [], [], []

        for _, pair in model_pred.groupby(["subject", "exercise"]):
            repeat_curves, repeat_aucs, repeat_matrices = [], [], []

            for unit, repeat in pair.groupby("unit"):
                y = repeat["y_true"].to_numpy()
                p = repeat["p_correct"].to_numpy()

                if np.unique(y).size != 2:
                    raise ValueError(f"{model}/{unit} needs both classes.")

                fpr, tpr, _ = roc_curve(y, p, drop_intermediate=False)
                repeat_curves.append(np.interp(fpr_grid, fpr, tpr))
                repeat_aucs.append(roc_auc_score(y, p))
                repeat_matrices.append(
                    confusion_matrix(y, repeat["y_hat"], labels=[0, 1], normalize="true")
                )

            pair_curves.append(np.mean(repeat_curves, axis=0))
            pair_aucs.append(np.mean(repeat_aucs))
            pair_matrices.append(np.mean(repeat_matrices, axis=0))

        if not pair_aucs:
            raise ValueError(f"No individualized predictions found for {model}.")

        mean_tpr = np.mean(pair_curves, axis=0)
        mean_auc = np.mean(pair_aucs)
        roc_ax.plot(
            np.r_[0, fpr_grid], np.r_[0, mean_tpr], color=COLORS[model],
            label=f"{MODEL_LABELS[model]} (mean AUC {mean_auc:.3f})",
        )

        mean_cm = np.mean(pair_matrices, axis=0)
        cm_ax.imshow(mean_cm, cmap="Blues", vmin=0, vmax=1)
        for row in range(2):
            for col in range(2):
                value = mean_cm[row, col]
                cm_ax.text(
                    col, row, f"{value:.1%}", ha="center", va="center",
                    color="white" if value > 0.6 else "black",
                )

        cm_ax.set(
            xticks=[0, 1], yticks=[0, 1],
            xticklabels=["Incorrect", "Correct"], yticklabels=["Incorrect", "Correct"],
            xlabel="Predicted", ylabel="True", title=MODEL_LABELS[model],
        )

    roc_ax.plot([0, 1], [0, 1], "--", color="#999999", linewidth=1)
    roc_ax.set(
        xlabel="False positive rate", ylabel="True positive rate",
        title="Individualized ROC averaged across pairs",
    )
    roc_ax.legend(frameon=False)
    cm_fig.suptitle("Individualized confusion matrices averaged across pairs (row %)")

    figure_dir = individual_dir / "figures"
    save_figure(roc_fig, figure_dir / "roc_pair_macro")
    save_figure(cm_fig, figure_dir / "confusion_pair_macro")
    print("Saved individualized figures in", figure_dir)

Saved individualized figures in /content/drive/MyDrive/NeuroYOLO_IRDS (1)/clean_results/intellirehabds_three_regime_v2_gpu/individualized/figures


### Export manuscript tables

The `manuscript/` folder consolidates all regimes, evaluation units, paired comparisons, and selected parameters. Gesture-specific tables and paired metrics remain in each regime's folder. ROC curves and confusion matrices are saved as PNG and PDF.

The cross-subject fold tests are exploratory. The individualized sensitivity analysis accounts for the fact that one participant can contribute multiple pairs. This rerun changes individualized tuning and holdout assignment, so use the new estimates and Methods together.

In [ ]:
export_manuscript_tables({name: table for name, table in results.items() if table is not None})

Manuscript tables: /content/drive/MyDrive/NeuroYOLO_IRDS (1)/clean_results/intellirehabds_three_regime_v2_gpu/manuscript


### Supplement · Model families and temporal order

Run the same fixed RF, SVM, XGBoost, and PyTorch LSTM configurations as in the new REHAB24-6 notebook. Compare original and temporally shuffled inputs using the generalized folds and the first ten individualized holdouts per pair in the full profile. The same permutation per recording is reused across models. Supplementary CSVs include split scores, pair means, shuffle deltas, and a participant-level sensitivity analysis with Holm adjustment.

Set `RUN_CROSS_METHOD = False` in setup for a primary-only run.

In [ ]:
# Fixed supplementary models and temporal shuffle experiment

CROSS_MODELS = ("RF", "SVM", "XGBoost", "LSTM")


class BaselineLSTM(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(len(ANGLE_NAMES), 64, batch_first=True)
        self.head = nn.Sequential(nn.Dropout(0.30), nn.Linear(64, 1))

    def forward(self, x):
        _, (hidden, _) = self.lstm(x)
        return self.head(hidden[-1]).squeeze(-1)


def cross_fit_predict(name, x_train, y_train, x_test, seed):
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.svm import SVC

    if name == "LSTM":
        seed_torch(seed)
        mean = x_train.mean(axis=(0, 1), keepdims=True)
        sd = x_train.std(axis=(0, 1), keepdims=True)
        sd = np.where(sd < 1e-8, 1.0, sd)
        train = torch.as_tensor((x_train - mean) / sd, dtype=torch.float32, device=DEVICE)
        test = torch.as_tensor((x_test - mean) / sd, dtype=torch.float32, device=DEVICE)
        target = torch.as_tensor(y_train, dtype=torch.float32, device=DEVICE)
        weights = len(y_train) / (2 * np.bincount(y_train, minlength=2))
        weights = torch.as_tensor(weights[y_train], dtype=torch.float32, device=DEVICE)
        model = BaselineLSTM().to(DEVICE)
        opt = torch.optim.Adam(model.parameters(), lr=1e-3)
        for _ in range(SETTINGS["cross_method_lstm_epochs"]):
            model.train()
            for start in range(0, len(train), 32):
                sl = slice(start, start + 32)
                opt.zero_grad(set_to_none=True)
                loss = F.binary_cross_entropy_with_logits(model(train[sl]), target[sl], weight=weights[sl])
                loss.backward()
                opt.step()
        model.eval()
        with torch.inference_mode():
            score = torch.cat([torch.sigmoid(model(batch)) for batch in test.split(128)]).cpu().numpy()
        return (score >= 0.5).astype(int), score, "probability"

    train, test = x_train.reshape(len(x_train), -1), x_test.reshape(len(x_test), -1)
    if name == "RF":
        model = RandomForestClassifier(n_estimators=SETTINGS["cross_method_trees"], class_weight="balanced",
                                       n_jobs=min(4, os.cpu_count() or 1), random_state=seed)
    elif name == "SVM":
        model = make_pipeline(StandardScaler(), SVC(kernel="rbf", C=1.0, gamma="scale", class_weight="balanced"))
    elif name == "XGBoost":
        from xgboost import XGBClassifier
        n0, n1 = np.bincount(y_train, minlength=2)
        model = XGBClassifier(n_estimators=SETTINGS["cross_method_trees"], max_depth=5, learning_rate=0.05,
                              subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0, min_child_weight=1.0,
                              objective="binary:logistic", eval_metric="logloss", tree_method="hist",
                              scale_pos_weight=float(n0 / n1), n_jobs=min(4, os.cpu_count() or 1), random_state=seed)
    else:
        raise KeyError(name)
    model.fit(train, y_train)
    if name == "SVM":
        score = model.decision_function(test)
        return (score >= 0).astype(int), score, "decision_score"
    score = model.predict_proba(test)[:, 1]
    return (score >= 0.5).astype(int), score, "probability"


def holm_adjust(p):
    p = np.asarray(p, dtype=float)
    out = np.full_like(p, np.nan)
    order = np.flatnonzero(np.isfinite(p))
    order = order[np.argsort(p[order])]
    if len(order):
        out[order] = np.minimum(1, np.maximum.accumulate(p[order] * np.arange(len(order), 0, -1)))
    return out


def run_cross_method(meta, angles, plan):
    if not RUN_CROSS_METHOD:
        return
    folder = RUN_DIR / "supplement_cross_method"
    y = meta.y.to_numpy()
    rng = np.random.default_rng(stable_seed("temporal_shuffle"))
    # A single independent permutation per repetition. Reused by every fit.
    permutations = np.stack([rng.permutation(T) for _ in range(len(angles))])
    shuffled = angles[np.arange(len(angles))[:, None], permutations]
    save_npz(folder / "temporal_permutations.npz", trial_id=meta.trial_id.to_numpy(dtype=str),
             frame_permutations=permutations)
    selected = [*plan["generalized"], *[s for s in plan["individualized"]
                                        if s.repeat <= SETTINGS["cross_method_repeats"]]]
    if not selected:
        print("No eligible cross-method units.")
        return
    rows, preds = [], []
    for split in tqdm(selected, desc="Fixed-config cross-method validation"):
        for model in CROSS_MODELS:
            for condition, tensor in (("original", angles), ("shuffled", shuffled)):
                path = folder / "checkpoints" / split.regime / split.unit / f"{model}_{condition}.npz"
                seed = stable_seed("cross_method", split.unit, model)
                if not path.exists():
                    try:
                        h, scores, score_type = cross_fit_predict(
                            model, tensor[split.train], y[split.train], tensor[split.test], seed,
                        )
                        save_npz(path, predicted=h, scores=scores, score_type=np.array(score_type),
                                 test_index=split.test)
                    except Exception as error:
                        raise RuntimeError(f"Cross-method {split.unit}/{model}/{condition} failed.") from error
                with np.load(path, allow_pickle=False) as payload:
                    if not np.array_equal(payload["test_index"], split.test):
                        raise RuntimeError(f"Split mismatch in {path}")
                    h, scores, score_type = payload["predicted"], payload["scores"], payload["score_type"].item()
                tag = {**split.keys(), "model": model, "condition": condition, "seed": seed}
                rows.append({**tag, **binary_metrics(y[split.test], scores, predicted=h)})
                trial = meta.iloc[split.test][["trial_id", "subject", "exercise", "y"]].copy()
                trial = trial.rename(columns={"y": "y_true"})
                preds.append(trial.assign(regime=split.regime, unit=split.unit, model=model, condition=condition,
                                          score=scores, score_type=score_type, y_hat=h))
    split_scores = pd.DataFrame(rows)
    generalized = split_scores[split_scores.regime == "generalized"]
    indiv = split_scores[split_scores.regime == "individualized"]
    pairs = indiv.groupby(["regime", "subject", "exercise", "model", "condition"], as_index=False)[list(METRICS)].mean()
    primary_units = pd.concat([generalized, pairs], ignore_index=True)
    summary = group_summary(primary_units, ["regime", "model", "condition"], METRICS)
    diff_frames, tests, participants = [], [], []
    for (regime, model), frame in primary_units.groupby(["regime", "model"], sort=False):
        keys = ["unit"] if regime == "generalized" else ["subject", "exercise"]
        wide = frame.pivot(index=keys, columns="condition", values="balanced_accuracy").dropna().reset_index()
        wide["shuffled_minus_original"] = wide["shuffled"] - wide["original"]
        diff_frames.append(wide.assign(regime=regime, model=model))
        tests.append({"regime": regime, "model": model,
                      **delta_summary(wide.shuffled_minus_original, groups=wide.get("subject"))})
        if regime == "individualized":
            values = wide.groupby("subject").shuffled_minus_original.mean()
            participants.append({"model": model, "n_subjects": len(values), "mean_delta": values.mean(),
                                 "median_delta": values.median(), "sign_flip_p": exact_sign_flip_p(values)})
    part = pd.DataFrame(participants)
    if not part.empty:
        part["holm_p"] = holm_adjust(part.sign_flip_p)
    for name, table in {
        "split_metrics": split_scores, "predictions": pd.concat(preds, ignore_index=True),
        "individualized_pair_metrics": pairs, "metric_summary": summary,
        "shuffle_deltas": pd.concat(diff_frames, ignore_index=True), "shuffle_comparisons": pd.DataFrame(tests),
        "shuffle_participant_sensitivity": part,
    }.items():
        save_csv(table, folder / f"{name}.csv")
    display(summary.loc[summary.metric == "balanced_accuracy"].round(3))

In [ ]:
run_cross_method(meta, X, plan)
print(f"Results available in {RUN_DIR}")

Fixed-config cross-method validation:   0%|          | 0/165 [00:00<?, ?it/s]

,regime,model,condition,metric,n,mean,sd,median,minimum,maximum
2,generalized,LSTM,original,balanced_accuracy,5,0.705,0.031,0.700,0.662,0.746
9,generalized,LSTM,shuffled,balanced_accuracy,5,0.680,0.071,0.676,0.607,0.764
16,generalized,RF,original,balanced_accuracy,5,0.633,0.076,0.637,0.511,0.708
23,generalized,RF,shuffled,balanced_accuracy,5,0.592,0.052,0.567,0.541,0.657
30,generalized,SVM,original,balanced_accuracy,5,0.731,0.037,0.724,0.685,0.786
37,generalized,SVM,shuffled,balanced_accuracy,5,0.630,0.015,0.632,0.610,0.647
44,generalized,XGBoost,original,balanced_accuracy,5,0.714,0.048,0.706,0.656,0.769
51,generalized,XGBoost,shuffled,balanced_accuracy,5,0.670,0.069,0.667,0.587,0.765
58,individualized,LSTM,original,balanced_accuracy,16,0.766,0.206,0.777,0.408,1.000
65,individualized,LSTM,shuffled,balanced_accuracy,16,0.819,0.178,0.873,0.383,1.000


Results available in /content/drive/MyDrive/NeuroYOLO_IRDS (1)/clean_results/intellirehabds_three_regime_v2_gpu


### Package the manuscript outputs


In [ ]:
# Package the manuscript outputs after the run has finished.
from zipfile import ZipFile, ZIP_DEFLATED

required = ["manuscript/metric_summary_all_regimes.csv"]
for regime, splits in plan.items():
    if splits:
        required.extend([f"{regime}/unit_metrics.csv", f"{regime}/predictions.csv"])
if plan["generalized"]:
    required.extend([
        "generalized/mqm/mqm_heldout_repetitions.csv",
        "generalized/robustness/severity_summary.csv",
    ])
if RUN_CROSS_METHOD and (plan["generalized"] or plan["individualized"]):
    required.extend([
        f"supplement_cross_method/{name}.csv" for name in (
            "split_metrics", "predictions", "individualized_pair_metrics", "metric_summary",
            "shuffle_deltas", "shuffle_comparisons", "shuffle_participant_sensitivity",
        )
    ])
missing = [name for name in required if not (RUN_DIR / name).is_file()]
if missing:
    raise FileNotFoundError("Complete the corresponding sections before packaging: " + ", ".join(missing))

destination = RUN_DIR / "manuscript_outputs.zip"
suffixes = {".csv", ".pdf", ".png", ".json", ".log"}
with atomic_output(destination) as temp:
    with ZipFile(temp, "w", compression=ZIP_DEFLATED) as archive:
        for folder, directories, files in os.walk(RUN_DIR):
            directories[:] = sorted(d for d in directories if d != "checkpoints")
            for name in sorted(files):
                path = Path(folder) / name
                if not name.startswith(".") and path.suffix.lower() in suffixes:
                    archive.write(path, path.relative_to(RUN_DIR))
    with ZipFile(temp) as archive:
        if archive.testzip() is not None or set(required) - set(archive.namelist()):
            raise RuntimeError("The results ZIP did not pass verification.")
print(f"Results ZIP ready: {destination}")

Results ZIP ready: /content/drive/MyDrive/NeuroYOLO_IRDS (1)/clean_results/intellirehabds_three_regime_v2_gpu/manuscript_outputs.zip
